# B2 — 600 pair tezgâhı (v6, 16 Eki)
Ressam (73K parametre, QAT8) + pair başına notlar (1/64, 1/32, 1/16 ızgaraları, entropi kodlu).

**Kurallar:** karar toplam skorla (8 bit ressamla ölçülen dist), karşılaştırma eşit KB'ta · 1 KB ≈ 0.00067 skor · sadece `SON` satırı okunur · her koşu yeni etiket.

**Çalıştırma:** Bölüm 1–8 (yalnız tanımlar, eğitim yok) → Bölüm 9'da tek deney hücresi.

| En iyi (15 Eki) | Ressam | 8 bit dist | Notlar | Ressam | Toplam |
|---|---|---|---|---|---|
| `pilot_rate_p600u_genis_qat8_lr15.pt` | [20,20,20,20,16,16,10] | 0.2325 (seg 0.2021) | 51.9 KB | 67.4 KB | **0.312** |

Tüm geçmiş: proje notu `gelistirme-ozeti-2026-10-15.md`. Eski v4 notebook'u Drive'da duruyor.

**v6:** zamansal entropi + delta kod (böl. 12, kayıpsız) · seviye raporu / kanal cerrahisi (böl. 13) · seg odaklı l7 + zor-pair örnekleme (böl. 14). Deneyler böl. 15'te.

## 1. Kurulum

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| cihaz:", device)
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, f"| bellek: {props.total_memory / 2**30:.1f} GB")

ROOT = Path("/content/drive/MyDrive/deneme")
VIDEO_PATH = Path("/content/drive/MyDrive/0.mkv")
MODEL_DIR = ROOT / "models"
TARGET_CACHE = ROOT / "checkpoints" / "hnerv384_targets_evalrgb_v1.pt"
OUT_DIR = ROOT / "b2"                      # bu notebook'un bütün çıktıları buraya
OUT_DIR.mkdir(exist_ok=True)

for p in (VIDEO_PATH, MODEL_DIR / "posenet.safetensors", MODEL_DIR / "segnet.safetensors", TARGET_CACHE):
    print("OK " if p.exists() else "YOK", p)

Mounted at /content/drive
PyTorch: 2.11.0+cu130 | cihaz: cuda
GPU: NVIDIA A100-SXM4-40GB | bellek: 39.5 GB
OK  /content/drive/MyDrive/0.mkv
OK  /content/drive/MyDrive/deneme/models/posenet.safetensors
OK  /content/drive/MyDrive/deneme/models/segnet.safetensors
OK  /content/drive/MyDrive/deneme/checkpoints/hnerv384_targets_evalrgb_v1.pt


In [ ]:
import gc, sys, traceback, types, warnings, torch, torch.nn as nn

KORU = {"distortion_net", "segnet", "posenet", "seg_targets", "pose_targets", "PILOT", "device"}  # silinmeyecekler

def _gpu_tutuyor(o, d=0):
    if torch.is_tensor(o): return o.is_cuda
    if isinstance(o, nn.Module): return any(t.is_cuda for t in [*o.parameters(), *o.buffers()])
    if isinstance(o, torch.optim.Optimizer):
        return any(torch.is_tensor(v) and v.is_cuda for s in o.state.values() for v in s.values()) \
            or any(p.is_cuda for g in o.param_groups for p in g["params"])
    if d < 2 and isinstance(o, dict): return any(_gpu_tutuyor(v, d + 1) for v in list(o.values())[:500])
    if d < 2 and isinstance(o, (list, tuple)): return any(_gpu_tutuyor(v, d + 1) for v in list(o)[:500])
    return False

def _kareleri_bosalt(exc, gorulen=None):
    gorulen = gorulen or set()
    while exc is not None and id(exc) not in gorulen:            # hata zinciri: cause / context
        gorulen.add(id(exc))
        if exc.__traceback__ is not None:
            traceback.clear_frames(exc.__traceback__)            # karelerdeki yerel değişkenleri sil
        _kareleri_bosalt(exc.__cause__, gorulen)
        exc = exc.__context__

def gpu_temizle(*isimler, koru=()):
    """gpu_temizle() → GPU tutan tüm globalleri sil (KORU hariç). İsim verirsen onları da siler."""
    try: torch.cuda.synchronize()                                # yarıda kalan GPU işini bitir
    except Exception: pass
    # 1) donmuş traceback kareleri (Durdur / OOM / hata)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                          # torch'un eski isimleri uyarı basmasın
        for o in gc.get_objects():                               # kim saklarsa saklasın: tüm traceback'ler
            try:
                if isinstance(o, types.TracebackType):
                    traceback.clear_frames(o)                    # karelerin yerellerini boşalt
            except ReferenceError:                               # ölü weakref proxy: dokununca patlıyor, atla
                continue
    ip = get_ipython()
    hatalar = [getattr(sys, "last_exc", None), getattr(sys, "last_value", None)]
    if getattr(ip, "last_execution_result", None) is not None:
        hatalar.append(ip.last_execution_result.error_in_exec)
        ip.last_execution_result.error_in_exec = None
    for e in hatalar:
        if isinstance(e, BaseException): _kareleri_bosalt(e)
    if getattr(sys, "last_traceback", None) is not None:
        traceback.clear_frames(sys.last_traceback)
    sys.last_exc = sys.last_type = sys.last_value = sys.last_traceback = None
    # 2) globaller: verilen isimler + GPU tutan her şey (korunanlar hariç)
    g = ip.user_ns
    sil = set(isimler) | {n for n, v in list(g.items())
                          if not n.startswith("_") and n not in KORU | set(koru) and _gpu_tutuyor(v)}
    for n in sorted(sil):
        g.pop(n, None)
    print("silindi:", ", ".join(sorted(sil)) or "—")
    # 3) Out[] geçmişi ve _ / __ / ___
    ip.run_line_magic("reset", "-f out")
    # 4) topla ve rafı geri ver
    gc.collect()
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()


def gpu_durum(etiket=""):
    a, r = torch.cuda.memory_allocated() / 2**30, torch.cuda.memory_reserved() / 2**30
    t = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"[GPU {etiket}] kullanılan {a:.1f} GB | ayrılmış {r:.1f} GB | toplam {t:.1f} GB")

## 2. Değerlendiriciler ve hedefler (SegNet / PoseNet donmuş)

In [ ]:
!uv pip -q install segmentation-models-pytorch timm einops safetensors av

import sys
sys.path.insert(0, str(ROOT))                      # research/ klasörü Drive'da, deneme/ altında
from research.modules import DistortionNet
from research.oracle_pair import seg_preprocess_grad, pose_preprocess_grad

distortion_net = DistortionNet().eval().to(device)
distortion_net.load_state_dicts(MODEL_DIR / "posenet.safetensors", MODEL_DIR / "segnet.safetensors", device)
distortion_net.requires_grad_(False)               # ağırlıkları asla değişmeyecek

n_seg = sum(p.numel() for p in distortion_net.segnet.parameters())
n_pose = sum(p.numel() for p in distortion_net.posenet.parameters())
print(f"SegNet {n_seg / 1e6:.1f}M | PoseNet {n_pose / 1e6:.1f}M parametre (donmuş)")

SegNet 9.5M | PoseNet 13.9M parametre (donmuş)


In [ ]:
targets = torch.load(TARGET_CACHE, map_location="cpu", weights_only=True)
seg_targets = targets["seg_class"].to(device, dtype=torch.uint8)    # [600, 384, 512]: piksel başına sınıf
pose_targets = targets["pose"].to(device, dtype=torch.float32)      # [600, 6]: pair başına kamera hareketi
NUM_PAIRS = seg_targets.shape[0]
print("Seg:", tuple(seg_targets.shape), "| Pose:", tuple(pose_targets.shape), "| pair:", NUM_PAIRS)

counts = torch.bincount(seg_targets.flatten().long(), minlength=5)
for c, n in enumerate(counts.tolist()):
    print(f"  class {c}: %{100 * n / counts.sum().item():6.2f}")
print("Pose std (boyut başına):", [round(v, 4) for v in pose_targets.std(0).tolist()])

Seg: (600, 384, 512) | Pose: (600, 6) | pair: 600
  class 0: % 23.23
  class 1: %  0.59
  class 2: % 49.52
  class 3: %  1.24
  class 4: % 25.43
Pose std (boyut başına): [1.2574, 0.0358, 0.0299, 0.0096, 0.0074, 0.0286]


## 3. Skor (resmi formül, FP32, TF32 kapalı)

In [ ]:

import torch.nn.functional as F

def inflate(native):
    """[B, 2, 3, 384, 512] → [B, 2, 3, 874, 1164]. İleri geçişte yuvarlar, geri geçişte gradyanı geçirir."""
    b = native.shape[0]
    up = F.interpolate(native.flatten(0, 1), size=(874, 1164), mode="bicubic", align_corners=False)
    up = up.clamp(0, 255)
    up = up + (up.round() - up).detach()          # STE: ileri = round, geri = özdeşlik
    return up.view(b, 2, 3, 874, 1164)


@torch.no_grad()
def score(render, ids_all, bs=16):
    """render(ids) → [B, 2, 3, 384, 512] (0–255). Resmi ölçümle aynı ayar: FP32, TF32 kapalı."""
    old = (torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32)
    torch.backends.cuda.matmul.allow_tf32 = torch.backends.cudnn.allow_tf32 = False
    wrong, pix, pose_sq = 0, 0, 0.0
    for s in range(0, len(ids_all), bs):
        ids = ids_all[s:s + bs]
        full = inflate(render(ids))
        pred = distortion_net.segnet(seg_preprocess_grad(full)).argmax(1)
        pose = distortion_net.posenet(pose_preprocess_grad(full))["pose"][:, :6].float()
        wrong += int((pred != seg_targets[ids].long()).sum())
        pix += pred.numel()
        pose_sq += float((pose - pose_targets[ids]).square().mean(1).sum())
    torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32 = old
    seg, pose_t = 100 * wrong / pix, (10 * pose_sq / len(ids_all)) ** 0.5
    return {"dist": seg + pose_t, "seg": seg, "pose": pose_t}


PILOT = torch.arange(120, device=device)         # test tezgâhı: ilk 120 pair (tam video 600)

## 4. Model ve loss

In [ ]:
import math
import torch.nn as nn
GRID_LEVELS = {64:2, 32:2, 16:2}
WIDTHS = [16, 16, 16, 16, 12, 12, 8]

class GridDecoder(nn.Module):
  def __init__(self, n_pairs, levels = GRID_LEVELS, widths = WIDTHS):
    super().__init__()
    self.grids = nn.ParameterDict({str(d): nn.Parameter(torch.zeros(n_pairs, c, 384 // d, 512 // d))
                                  for d, c in levels.items()})

    self.inject = nn.ModuleDict({str(d): nn.Conv2d(c, widths[int(math.log2(64 // d))], 1)
                                     for d, c in levels.items()})
    self.ups = nn.ModuleList(nn.Conv2d(widths[i], 4 * widths[i + 1], 3, padding=1) for i in range(6))
    self.skips = nn.ModuleList(nn.Conv2d(widths[i], widths[i + 1], 1) for i in range(6))
    self.head = nn.Conv2d(widths[-1], 6, 3, padding=1)
  def forward(self, ids, grids = None):
    g = grids if grids is not None else {k: v[ids] for k, v in self.grids.items()}
    x = self.inject["64"](g["64"])
    for i in range(6):
      up = F.pixel_shuffle(self.ups[i](x), 2)
      skip = F.interpolate(self.skips[i](x), scale_factor = 2, mode = "bilinear", align_corners = False)
      x = F.gelu(up + skip)
      d = str(64//2 ** (i+1))
      if d in self.inject:
        x = x + self.inject[d](g[d])
    return (torch.sigmoid(self.head(x)) * 255).view(len(ids), 2, 3, 384, 512)

In [ ]:

def l7_loss(logits, truth, tau = 0.3,thr =1.0, mult = 4.0):
  correct = logits.gather(1, truth.unsqueeze(1))
  others = logits.clone()
  others.scatter_(1, truth.unsqueeze(1), torch.finfo(others.dtype).min)
  margin = correct - others.amax(1, keepdim = True)
  per_pixel = tau * F.softplus(-margin / tau)

  with torch.no_grad():
    w = 1.0 + mult * (margin < thr).float()
    w = w/w.mean()

  return (per_pixel * w).mean()

def task_loss(native, ids):
  full = inflate(native)
  logits = distortion_net.segnet(seg_preprocess_grad(full))
  pose = distortion_net.posenet(pose_preprocess_grad(full))["pose"][:, :6].float()
  seg_l = 100 * l7_loss(logits, seg_targets[ids].long())
  pose_l = torch.sqrt(10 * F.mse_loss(pose, pose_targets[ids]) + 1e-6)
  return seg_l + pose_l, seg_l.item(), pose_l.item()

## 5. Bit / boyut araçları ve yuvarlama (sert, yumuşak, C3)

In [ ]:
BIT2SCORE = 25 / 8 / 37545489                     # 1 bitin skordaki karşılığı
NET_KB = 45                                       # ressam, 8 bit (entropi kodu yalnızca 44.1 KB'a indiriyor)
rate_of_kb = lambda kb: 25 * kb * 1e3 / 37545489  # KB → skor


@torch.no_grad()
def quant_net_(m, bits=8):
    """Ressamın conv ağırlıklarını yerinde, kanal başına simetrik `bits` bite yuvarlar."""
    qmax = 2 ** (bits - 1) - 1
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            s = mod.weight.abs().amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-12) / qmax
            mod.weight.copy_(torch.round(mod.weight / s).clamp(-qmax, qmax) * s)


def entropy_kb(q):
    """Tamsayı dizisinin ampirik entropisi (değerler bağımsız) → KB."""
    _, counts = torch.unique(q, return_counts=True)
    p = counts.double() / q.numel()
    return float(-(counts * p.log2()).sum()) / 8 / 1e3


def laplace_bits(y, b):
    """y değerinin (±0.5 kutusu) Laplace(0, b) altında bit maliyeti.
    DÜZELTME (6 Eki): p_in'deki işaret '+' olmalı; eski sürümde '-' idi → sıfıra yakın değerler 0 bit sayılıyordu."""
    a = y.abs()
    a_in, a_out = a.clamp(max=0.5), a.clamp(min=0.5)
    p_in = 1 - 0.5 * (torch.exp(-(0.5 - a_in) / b) + torch.exp(-(0.5 + a_in) / b))
    p_out = 0.5 * (torch.exp(-(a_out - 0.5) / b) - torch.exp(-(a_out + 0.5) / b))
    return -torch.log2(torch.where(a < 0.5, p_in, p_out).clamp_min(1e-9))


def grids_for(m, ids, train):
    """Eğitimde: ileri geçiş yuvarlanmış değeri görür (STE) + bit tahmini için gürültülü kopya.
       Değerlendirmede: düz yuvarlama."""
    fwd, noisy = {}, {}
    for k, g in m.grids.items():
        x = g[ids]
        fwd[k] = x + (torch.round(x) - x).detach() if train else torch.round(x)
        noisy[k] = x + torch.rand_like(x) - 0.5
    return fwd, noisy


def soft_round(y, alpha):
    """alpha küçük → neredeyse y (düz çizgi); alpha büyük → round(y) (merdiven)."""
    m = torch.floor(y) + 0.5                       # bulunduğu kutunun ortası (ör. 0.3 → 0.5)
    r = y - m                                      # kutu içindeki konum: -0.5 .. +0.5
    return m + torch.tanh(alpha * r) / (2 * math.tanh(alpha / 2))


def grids_for_soft(m, ids, alpha):
    """Eğitimde yumuşak yuvarlama: ressam s(s(y) + gürültü)'yü görür, bit tahmini s(y) + gürültü'den."""
    fwd, noisy = {}, {}
    for k, g in m.grids.items():
        s = soft_round(g[ids], alpha)
        noisy[k] = s + torch.rand_like(s) - 0.5
        fwd[k] = soft_round(noisy[k], alpha)
    return fwd, noisy


def kumaraswamy_noise(shape, a, device):
    """[-0.5, 0.5] aralığında gürültü. a = 1 → düz (bizim eski ±0.5); a > 1 → ortaya yığılmış."""
    b = (2 ** a * (a - 1) + 1) / a                       # tepe tam 0.5'te olsun diye b seçilir
    u = torch.rand(shape, device=device).clamp(1e-6, 1 - 1e-6)
    x = (1 - (1 - u) ** (1 / b)) ** (1 / a)              # ters CDF ile örnekleme → [0, 1]
    return x - 0.5                                       # → [-0.5, 0.5]


def grids_for_c3(m, ids, alpha, a):
    """C3 1. aşama: bit tahmini s(y) + gürültü'den, ressam s(s(y) + gürültü)'yü görür."""
    fwd, noisy = {}, {}
    for k, g in m.grids.items():
        s = soft_round(g[ids], alpha)
        noisy[k] = s + kumaraswamy_noise(s.shape, a, s.device)
        fwd[k] = soft_round(noisy[k], alpha)
    return fwd, noisy

## 6. Bağlamlı entropi modeli (C3 A.1.4 tarzı)
Her not için komşularından (5×5 nedensel maske) ve bir kaba seviyeden μ, σ tahmini → Laplace bit. Sıra 1/64 → 1/32 → 1/16. Şimdilik **sonradan** uygulanıyor (birlikte eğitim notları kötüleştirdi).

In [ ]:
class MaskedConv2d(nn.Conv2d):
  def __init__(self, c_in, c_out, k = 5):
    super().__init__(c_in, c_out, k, padding = k//2)
    mask = torch.ones(k, k)
    mask[k // 2, k // 2:] = 0
    mask[k // 2 + 1:,:] = 0
    self.register_buffer("mask", mask.view(1, 1, k, k))
  def forward(self, x):
    return F.conv2d(x, self.weight * self.mask, self.bias, padding = self.padding)

In [ ]:
class LevelEntropy(nn.Module):
  def __init__(self, c = 2, h = 16, has_coarse = True):
    super().__init__()
    self.same = MaskedConv2d(c, h, 5)
    self.coarse = nn.Conv2d(c, h, 1) if has_coarse else None
    self.mix = nn.Sequential(
        nn.GELU(), nn.Conv2d(h, h, 1),
        nn.GELU(), nn.Conv2d(h, 2 * c, 1),
    )
    nn.init.zeros_(self.mix[-1].weight)
    nn.init.zeros_(self.mix[-1].bias)
  def forward(self, q, q_coarse):
    x = self.same(q)
    if self.coarse is not None:
      up = F.interpolate(q_coarse, size=q.shape[-2:], mode="nearest")
      x = x + self.coarse(up)
    mu, log_s = self.mix(x).chunk(2, dim = 1)
    return mu, log_s.exp().clamp(0.1)


class ContextEntropy(nn.Module):
  ORDER = ["64", "32", "16"]
  def __init__(self, levels = GRID_LEVELS):
    super().__init__()
    self.nets = nn.ModuleDict({k: LevelEntropy(levels[int(k)], has_coarse=(i > 0))
                            for i, k in enumerate(self.ORDER)})
  def forward(self, q):
    out = {}
    for i, k in enumerate(self.ORDER):
      onceki = q[self.ORDER[i-1]] if i>0 else None
      out[k] = self.nets[k](q[k], onceki)
    return out

In [ ]:
ent_kb = lambda e: sum(p.numel() for p in e.parameters()) / 1e3        # entropi ağı ≈ parametre başına 1 bayt

def context_bits(ent, ctx, vals):
    """ctx: bağlam olarak kullanılan yuvarlanmış notlar. vals: biti ölçülen değerler
    (eğitimde gürültülü, ölçümde tamsayı). Döner: {seviye: toplam bit}."""
    tahmin = ent(ctx)
    return {k: laplace_bits(vals[k] - mu, s).sum() for k, (mu, s) in tahmin.items()}


@torch.no_grad()
def context_kb(ent, grids, ids_all, bs=20):
    """Tamsayı notların bağlamlı model altındaki kod uzunluğu, 600 pair'e ölçekli, seviye seviye KB."""
    toplam = {k: 0.0 for k in ContextEntropy.ORDER}
    for s in range(0, len(ids_all), bs):
        ids = ids_all[s:s + bs]
        q = {k: torch.round(grids[k][ids]) for k in ContextEntropy.ORDER}
        for k, b in context_bits(ent, q, q).items():
            toplam[k] += b.item()
    olcek = NUM_PAIRS / len(ids_all) / 8 / 1e3                       # bit → KB, 600 pair'e ölçekle
    return {k: v * olcek for k, v in toplam.items()}

## 7. Eğitim fonksiyonları

In [ ]:
import copy, time
from tqdm.auto import tqdm


class EMA:
    """Ağırlıkların hareketli ortalaması (#95: 0.999).
    Gölge kopya her adımda canlı modele biraz yaklaşır → yuvarlama gürültüsüyle zıplayan değerleri yumuşatır.
    ufuk_adim: gölgenin kabaca kaç adımı 'hatırladığı' (decay = 1 - 1/ufuk)."""
    def __init__(self, model, ufuk_adim):
        self.decay = 1 - 1 / max(1, ufuk_adim)
        self.golge = {k: v.detach().clone() for k, v in model.state_dict().items()}
        self.n = 0

    @torch.no_grad()
    def guncelle(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))          # başta hızlı ısınma
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.golge[k].lerp_(v, 1 - d)                       # golge = d·golge + (1-d)·v
            else:
                self.golge[k].copy_(v)

    def kopya(self, sablon):
        """Gölge ağırlıklarla yeni bir model (değerlendirme / kayıt için)."""
        m = copy.deepcopy(sablon)
        m.load_state_dict(self.golge)
        return m


def qat_ac(m, bits=8):
    """Ressamın conv ağırlıklarını geçici olarak 8 bit (kanal başına) yuvarlanmış halleriyle değiştirir.
    İleri ve geri geçiş yuvarlanmış ağırlığı görür; adım asıl (FP32) ağırlığa uygulanır (STE, #95 gibi).
    Dönen sözlüğü backward'dan SONRA qat_kapat'a ver."""
    qmax = 2 ** (bits - 1) - 1
    asil = {}
    for ad, mod in m.named_modules():
        if isinstance(mod, nn.Conv2d):
            w = mod.weight.data
            s = w.abs().amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-12) / qmax
            asil[ad] = w
            mod.weight.data = torch.round(w / s).clamp(-qmax, qmax) * s
    return asil


def qat_kapat(m, asil):
    for ad, mod in m.named_modules():
        if ad in asil:
            mod.weight.data = asil[ad]


def ressam_kb(m):
    """8 bit ressam ≈ parametre başına 1 bayt (entropi kodu biraz daha küçültür; temkinli tahmin)."""
    return sum(p.numel() for n, p in m.named_parameters() if not n.startswith("grids")) / 1e3


def fp32_sifirdan(tag, epochs=2400, batch=20, lr_net=1e-3, lr_grid=1e-2, warm=20, ema_ep=25, eval_every=400):
    """Rastgele başlangıçtan FP32 eğitimi (bit yok), PILOT pair'lerinde, GRID_LEVELS / WIDTHS mimarisiyle.
    Kayıt: pilot_{tag}.pt (EMA ağırlıkları). Test B'de iki mimariyi aynı başlangıç koşullarında kıyaslamak için."""
    torch.manual_seed(0)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    net_p = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    opt = torch.optim.Adam([{"params": net_p, "lr": lr_net},
                            {"params": list(m.grids.parameters()), "lr": lr_grid}])
    adim = math.ceil(len(PILOT) / batch)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    ema = EMA(m, ema_ep * adim) if ema_ep else None
    print(f"[{tag}] FP32 sıfırdan | {len(PILOT)} pair | ressam {ressam_kb(m) * 1e3:,.0f} parametre | {epochs} epoch")
    t0 = time.time()
    for ep in tqdm(range(epochs)):
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sched.step()
            if ema: ema.guncelle(m)
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            em = ema.kopya(m) if ema else m
            r = score(em, PILOT)
            print(f"[{tag}] E{ep + 1:04d} | dist {r['dist']:.4f} (seg {r['seg']:.4f}, pose {r['pose']:.4f}) "
                  f"| {(time.time() - t0) / 60:.1f} dk", flush=True)
            torch.save(em.state_dict(), OUT_DIR / f"pilot_{tag}.pt")
    return r


In [ ]:
def hazirla_rate(fp32_path, tag, step0=None, oran=0.57):
    """FP32 modeli bit aşamasına hazırlar. Adım = oran × notların std'si (ölçekten bağımsız birim).
    Pilotta 0.25 ≈ 0.57·std idi; 200 pair'de std 0.63 → adım ≈ 0.34."""
    sd = torch.load(fp32_path, map_location=device)
    if step0 is None:
        std = torch.cat([sd[f"grids.{k}"][PILOT].flatten() for k in GRID_LEVELS]).std().item()
        step0 = oran * std
    for k in GRID_LEVELS:
        sd[f"grids.{k}"] /= step0
        sd[f"inject.{k}.weight"] *= step0
    log_b = {str(k): torch.zeros(sd[f"grids.{k}"].shape[1], device=device) for k in GRID_LEVELS}
    yol = OUT_DIR / f"pilot_rate_{tag}_init.pt"
    torch.save({"model": sd, "log_b": log_b, "step0": step0}, yol)
    print(f"[{tag}] adım {step0:.3f} → {yol.name}")
    return yol


In [ ]:
def run_rate(lam_max, epochs, init_path, tag, ramp_ep=50, eval_every=50, lr_carpan=1.0, grid_carpan=1.0,
             lam_min=3.0, batch=8, warm=50, soft=None, b_min=0.3, ctx=False, kuma=None, ema_ep=None, birikim=1,
             sinif=None, kod_lr=1e-2):
    """birikim > 1: `birikim` parça (her biri `batch` pair) biriktirilip tek Adam adımı atılır.
    Etkin batch = batch × birikim; bellek yalnız `batch` kadar."""
    ck = torch.load(init_path, map_location=device)
    m = (sinif or GridDecoder)(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device); m.load_state_dict(ck["model"])

    log_b = nn.ParameterDict({k: nn.Parameter(v.detach().clone().to(device)) for k, v in ck["log_b"].items()})
    ent = ContextEntropy().to(device) if ctx else None
    if ctx and ck.get("ent") is not None:
        ent.load_state_dict(ck["ent"])
    net_p = [p for n, p in m.named_parameters() if not n.startswith(("grids", "codes"))]
    grid_p = list(m.grids.parameters())
    kod_p = [p for n, p in m.named_parameters() if n.startswith("codes")]
    gruplar = [{"params": net_p, "lr": 3e-4 * lr_carpan},
               {"params": grid_p, "lr": 3e-2 * lr_carpan * grid_carpan},
               {"params": list(log_b.parameters()), "lr": 1e-2 * lr_carpan}]
    if kod_p:
        gruplar.append({"params": kod_p, "lr": kod_lr})   # kodlar pair başına: ressam LR'siyle yerinden oynamıyor
    if ctx:
        gruplar.append({"params": list(ent.parameters()), "lr": 3e-3 * lr_carpan})
    opt = torch.optim.Adam(gruplar)
    adim = math.ceil(len(PILOT) / (batch * birikim))                   # epoch başına Adam adımı
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    kirp = net_p + grid_p + kod_p + (list(ent.parameters()) if ctx else [])
    ema = EMA(m, ema_ep * adim) if ema_ep else None
    print(f"[{tag}] bit aşaması | parça {batch} × birikim {birikim} = etkin batch {batch * birikim} "
          f"| {adim} adım/epoch | not LR {3e-2 * lr_carpan * grid_carpan:.2e} | ressam LR {3e-4 * lr_carpan:.2e}")
    for ep in tqdm(range(epochs)):
        lam = lam_min + (lam_max - lam_min) * min(1, (ep + 1) / ramp_ep)
        t = ep / max(1, epochs - 1)
        alpha = 1 / (1 / soft[0] + (1 / soft[1] - 1 / soft[0]) * t) if soft else None
        a_k = kuma[0] + (kuma[1] - kuma[0]) * t if kuma else None
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        parcalar = [perm[i:i + batch] for i in range(0, len(perm), batch)]
        for j in range(0, len(parcalar), birikim):
            grup = parcalar[j:j + birikim]
            n_grup = sum(len(p) for p in grup)
            opt.zero_grad(set_to_none=True)
            for ids in grup:                                           # gradyanlar birikir, adım yok
                if kuma:
                    fwd, noisy = grids_for_c3(m, ids, alpha, a_k)
                elif soft:
                    fwd, noisy = grids_for_soft(m, ids, alpha)
                else:
                    fwd, noisy = grids_for(m, ids, train=True)
                task, _, _ = task_loss(m(ids, grids=fwd), ids)
                if ctx:
                    bits = sum(context_bits(ent, fwd, noisy).values())
                else:
                    bits = sum(laplace_bits(noisy[k], log_b[k].exp().clamp_min(b_min).view(1, -1, 1, 1)).sum()
                               for k in noisy)
                # büyük batch'in kaybıyla birebir aynı: görev kaybı pair ortalaması, bitler grup toplamı / grup boyu
                loss = (task * len(ids) + lam * bits * NUM_PAIRS * BIT2SCORE) / n_grup
                loss.backward()
            torch.nn.utils.clip_grad_norm_(kirp, 1.0); opt.step(); sched.step()
            if ema: ema.guncelle(m)
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            em = ema.kopya(m) if ema else m
            r = score(lambda ids: em(ids, grids=grids_for(em, ids, train=False)[0]), PILOT)
            q = {k: torch.round(g[PILOT].detach()) for k, g in em.grids.items()}
            kb_iid = sum(entropy_kb(v) for v in q.values()) * NUM_PAIRS / len(PILOT)
            kb = (sum(context_kb(ent, {k: g.detach() for k, g in em.grids.items()}, PILOT).values()) + ent_kb(ent)
                  if ctx else kb_iid)
            sifir = "/".join(f"%{100 * (v == 0).float().mean():.0f}" for v in q.values())
            print(f"[{tag}] E{ep + 1:03d} λ={lam:.0f} | dist {r['dist']:.4f} | ızgara ~{kb:.0f} KB"
                  f" | sıfır {sifir} | toplam ≈ {r['dist'] + rate_of_kb(kb + NET_KB):.3f}{' [EMA]' if ema else ''}")
    em = ema.kopya(m) if ema else m
    torch.save({"model": em.state_dict(), "log_b": log_b.state_dict(), "step0": ck.get("step0"),
                "ent": ent.state_dict() if ctx else None}, OUT_DIR / f"pilot_rate_{tag}.pt")
    return r, kb


def net_ft(init_path, tag, epochs=200, batch=8, lr=3e-4, warm=10, ema_ep=None, qat=False, eval_every=50):
    """Notlar yuvarlanıp dondurulur, yalnız ressam eğitilir.
    v4: EMA (isteğe bağlı) · ressam QAT (isteğe bağlı) · her ölçümde 8 bit ressamla da ölçer ve
    8 bit dist'e göre EN İYİ checkpoint'i tutar (son epoch değil)."""
    ck = torch.load(init_path, map_location=device)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device); m.load_state_dict(ck["model"])
    with torch.no_grad():
        for g in m.grids.values(): g.round_()
    m.grids.requires_grad_(False)
    if ck.get("ent") is not None:
        ent = ContextEntropy().to(device); ent.load_state_dict(ck["ent"])
        kb = sum(context_kb(ent, dict(m.grids.items()), PILOT).values()) + ent_kb(ent)
    else:
        kb = sum(entropy_kb(m.grids[k][PILOT]) for k in m.grids) * NUM_PAIRS / len(PILOT)
    params = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    adim = math.ceil(len(PILOT) / batch)
    opt = torch.optim.Adam(params, lr=lr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    ema = EMA(m, ema_ep * adim) if ema_ep else None
    net_kb = ressam_kb(m)
    print(f"[{tag}] net_ft | batch {batch} | LR {lr:.2e} | EMA {'yok' if not ema else f'{ema_ep} epoch'} "
          f"| QAT {'8 bit' if qat else 'yok'} | notlar {kb:.0f} KB | ressam {net_kb:.0f} KB")
    en_iyi = {"d8": float("inf")}
    for ep in tqdm(range(epochs)):
        perm = PILOT[torch.randperm(len(PILOT), device=device)]
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            asil = qat_ac(m) if qat else None
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True); loss.backward()
            if qat: qat_kapat(m, asil)                         # backward'dan sonra asıl ağırlıklara dön
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step()
            if ema: ema.guncelle(m)
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            em = ema.kopya(m) if ema else copy.deepcopy(m)
            r = score(em, PILOT)
            sd = copy.deepcopy(em.state_dict())                 # FP32 hali (kayıt için)
            quant_net_(em, 8)
            r8 = score(em, PILOT)                               # gerçek arşivdeki gibi 8 bit ressam
            if r8["dist"] < en_iyi["d8"]:
                en_iyi = {"d8": r8["dist"], "r": r, "r8": r8, "ep": ep + 1, "sd": sd}
            del em
            print(f"[{tag}] E{ep + 1:03d} | dist {r['dist']:.4f} | 8 bit {r8['dist']:.4f} | ızgara {kb:.0f} KB "
                  f"| toplam {r8['dist'] + rate_of_kb(kb + net_kb):.3f} | en iyi {en_iyi['d8']:.4f} (E{en_iyi['ep']})")
    m.load_state_dict(en_iyi["sd"])
    r, r8 = en_iyi["r"], en_iyi["r8"]
    print(f"[{tag}] SON | en iyi E{en_iyi['ep']} | dist {r['dist']:.4f} | 8 bit {r8['dist']:.4f} "
          f"(seg {r8['seg']:.4f}, pose {r8['pose']:.4f}) | notlar {kb:.0f} KB + ressam {net_kb:.0f} KB "
          f"| TOPLAM {r8['dist'] + rate_of_kb(kb + net_kb):.3f}")
    torch.save({"model": en_iyi["sd"], "log_b": ck["log_b"], "step0": ck.get("step0"), "ent": ck.get("ent")},
               OUT_DIR / f"pilot_rate_{tag}.pt")
    return m, r, kb, r8


## 8. Tezgâh döngüsü
Koşu kuyruğu (`gece`) ve `mimari_kur`. Sonuçlar `tezgah_sonuc.json` + `tezgah_log.txt`.

In [ ]:
import sys, json, gc, time, traceback
from contextlib import redirect_stdout

LOG = OUT_DIR / "tezgah_log.txt"
SONUC = OUT_DIR / "tezgah_sonuc.json"
GECE = OUT_DIR / "gece_sonuc.json"                       # 600 pair gece kuyruğu (ayrı notebook'tan)
FP32_KAYNAK = "pilot_p600_uzun.pt"
VARSAYILAN = dict(levels={64: 2, 32: 2, 16: 2}, widths=[16, 16, 16, 16, 12, 12, 8])
REFERANS = {"p200u (ft900)":     ("sert", 30, 200, 0.2483, 94.6, 0.0898),
            "p600c3 (ft900)":    ("C3",   30, 600, 0.3033, 78.3, 0.0835),
            "p600sifir (ft900)": ("C3*",  30, 600, 0.3342, 65.1, 0.0835)}

BELLEK_GB = torch.cuda.get_device_properties(0).total_memory / 2 ** 30
B_BIT = 20
B_FT = 40 if BELLEK_GB > 30 else 20                      # L4 (22 GB) batch 40'ta OOM veriyordu
print(f"GPU belleği {BELLEK_GB:.0f} GB → bit aşaması batch {B_BIT}, net_ft batch {B_FT}")


class Tee:                                               # yazılanı hem ekrana hem dosyaya gönder
    def __init__(self, *akislar): self.akislar = akislar
    def write(self, s):
        for a in self.akislar: a.write(s); a.flush()
    def flush(self):
        for a in self.akislar: a.flush()


def mimari_kur(levels, widths, pairs):
    """Koşunun mimarisini ve pair sayısını globallere yazar (hazirla_rate / run_rate / net_ft bunları okur)."""
    global PILOT, GRID_LEVELS, WIDTHS, NET_KB
    PILOT = torch.arange(pairs, device=device)
    GRID_LEVELS, WIDTHS = dict(levels), list(widths)
    NET_KB = ressam_kb(GridDecoder(1, GRID_LEVELS, WIDTHS))


@torch.no_grad()
def fp32_ref(fp32):
    """Başlangıç FP32 modelinin aynı pair'lerdeki dist'i (oranın paydası)."""
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    m.load_state_dict(torch.load(OUT_DIR / fp32, map_location=device))
    d = score(m, PILOT)["dist"]
    del m; torch.cuda.empty_cache()
    return d


def gece_kosusu(T, lam, soft, kuma, pairs=120, epochs=1500, ramp=500, ft_ep=600,
                ema_ep=None, not_lr=1.0, qat=False, fp32=FP32_KAYNAK, fp32_ep=None,
                levels=None, widths=None, b_bit=None, b_ft=None):
    mimari_kur(levels or VARSAYILAN["levels"], widths or VARSAYILAN["widths"], pairs)
    print(f"\n[{T}] PILOT {pairs} pair | ızgaralar {GRID_LEVELS} | ressam {WIDTHS} ({NET_KB:.0f}K parametre)")
    if not (OUT_DIR / fp32).exists():
        if fp32_ep is None:
            raise FileNotFoundError(f"{fp32} yok ve fp32_ep verilmemiş")
        fp32_sifirdan(Path(fp32).stem.removeprefix("pilot_"), epochs=fp32_ep, batch=b_bit or B_BIT)
    ref = fp32_ref(fp32)
    print(f"[{T}] başlangıç {fp32}: FP32 dist {ref:.4f}")

    B = b_bit or B_BIT; s = (B / 8) ** 0.5
    if (OUT_DIR / f"pilot_rate_{T}.pt").exists():
        print(f"[{T}] bit aşaması zaten var, atlandı")
    else:
        init = hazirla_rate(OUT_DIR / fp32, T)
        run_rate(lam, epochs, init, T, ramp_ep=ramp, eval_every=250, batch=B, lr_carpan=s,
                 grid_carpan=0.55 / s * not_lr, b_min=0.3, soft=soft, kuma=kuma, ema_ep=ema_ep)

    B = b_ft or B_FT; s = (B / 8) ** 0.5
    _, r, kb, r8 = net_ft(OUT_DIR / f"pilot_rate_{T}.pt", f"{T}_ft{ft_ep}", epochs=ft_ep, batch=B,
                          lr=3e-4 * s, ema_ep=ema_ep, qat=qat)
    ayar = (("C3" if soft else "sert") + ("+ema" if ema_ep else "") + (f"+lr×{not_lr:.2g}" if not_lr != 1 else "")
            + ("+qat" if qat else "") + (f"+{len(GRID_LEVELS)}sv/{NET_KB:.0f}K" if levels or widths else ""))
    return {"ayar": ayar, "yontem": "C3" if soft else "sert", "lam": lam, "pairs": pairs,
            "dist": r["dist"], "dist8": r8["dist"], "seg8": r8["seg"], "pose8": r8["pose"],
            "kb": kb, "net_kb": NET_KB, "fp32_dist": ref, "toplam": r8["dist"] + rate_of_kb(kb + NET_KB)}


def ozet(sonuclar):
    satirlar = []
    for T, x in sonuclar.items():
        if T.startswith("_"): continue
        f = x.get("fp32_dist") or sonuclar.get(f"_fp32_{x['pairs']}", float("nan"))
        satirlar.append((T, x.get("ayar", x["yontem"]), x["lam"], x["pairs"], x["dist"], x.get("dist8"),
                         x["kb"], x.get("net_kb", 45), f))
    if GECE.exists():                                    # 600 pair gece kuyruğu (8 bit ölçümü yok)
        for T, x in json.loads(GECE.read_text()).items():
            satirlar.append((T, x["yontem"], x["lam"], 600, x["dist"], None, x["kb"], 45, 0.0835))
    satirlar += [(T, y, l, p, d, None, k, 45, f) for T, (y, l, p, d, k, f) in REFERANS.items()]
    toplam = lambda z: (z[5] if z[5] is not None else z[4]) + rate_of_kb(z[6] + z[7])
    print("\n=== ÖZET (toplama göre) ===")
    print(f"{'koşu':22s} {'ayar':20s} {'λ':>3s} {'pair':>4s} {'dist':>7s} {'8 bit':>7s} {'oran':>5s} "
          f"{'notKB':>6s} {'ağKB':>5s} {'toplam':>7s}")
    for z in sorted(satirlar, key=toplam):
        T, y, l, p, d, d8, k, nk, f = z
        d8s = f"{d8:7.4f}" if d8 is not None else f"{'–':>7s}"
        print(f"{T:22s} {y:20s} {l:3d} {p:4d} {d:7.4f} {d8s} {(d8 or d) / f:5.1f} {k:6.1f} {nk:5.0f} {toplam(z):7.3f}")
    print("toplam: 8 bit ölçümü varsa onunla, yoksa FP32 ressamlı dist ile (eski satırlar).")


def gece(sira):
    sonuclar = json.loads(SONUC.read_text()) if SONUC.exists() else {}
    with open(LOG, "a") as f, redirect_stdout(Tee(sys.stdout, f)):
        print(f"\n=== başladı {time.strftime('%Y-%m-%d %H:%M')} ===")
        for k in sira:
            if k["T"] in sonuclar:
                print(f"[{k['T']}] zaten bitmiş, atlandı"); continue
            t0 = time.time()
            try:
                sonuclar[k["T"]] = gece_kosusu(**k) | {"dk": round((time.time() - t0) / 60)}
                SONUC.write_text(json.dumps(sonuclar, indent=1))
                print(f"[{k['T']}] BİTTİ: {sonuclar[k['T']]}")
            except Exception:
                print(f"[{k['T']}] HATA:\n{traceback.format_exc()}")
            gc.collect(); torch.cuda.empty_cache()
        ozet(sonuclar)


C3 = dict(soft=(3.33, 10.0), kuma=(2.0, 1.0))
SERT = dict(soft=None, kuma=None)


GPU belleği 39 GB → bit aşaması batch 20, net_ft batch 40


## 9. Ölçüm, genişletme, QAT ince ayar, devam turları

In [ ]:
def ressam_kb_bit(sd, b):
    """quant_net_ formatı: conv ağırlıkları kanal başına simetrik b bit (entropi), bias + ölçekler fp16."""
    qmax, bit = 2 ** (b - 1) - 1, 0.0
    for k, v in sd.items():
        if k.startswith("grids"):
            continue
        v = v.float().cpu()
        if v.dim() == 4:
            s = v.abs().amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-12) / qmax
            kod = torch.round(v / s).clamp(-qmax, qmax).long().flatten()
            p = torch.bincount(kod + qmax).float(); p = p[p > 0] / kod.numel()
            bit += -(p * p.log2()).sum().item() * kod.numel() + 16 * v.shape[0]
        else:
            bit += 16 * v.numel()
    return bit / 8 / 1000


def bit_tarama(ad, widths, bitler=(8, 7, 6, 5)):
    mimari_kur(GRID_LEVELS, widths, 600)
    ck = torch.load(OUT_DIR / f"pilot_rate_{ad}.pt", map_location=device)
    for b in bitler:
        m = GridDecoder(NUM_PAIRS, GRID_LEVELS, widths).to(device)
        m.load_state_dict(ck["model"])
        with torch.no_grad():
            for g in m.grids.values(): g.round_()
            quant_net_(m, b)
            not_kb = sum(entropy_kb(m.grids[k][PILOT]) for k in m.grids)
            r = score(m, PILOT)
        r_kb = ressam_kb_bit(ck["model"], b)
        print(f"{ad:24s} | {b} bit | dist {r['dist']:.4f} (seg {r['seg']:.4f}) | "
              f"notlar {not_kb:.1f} + ressam {r_kb:.1f} KB | TOPLAM {r['dist'] + rate_of_kb(not_kb + r_kb):.3f}")
        del m; torch.cuda.empty_cache()


@torch.no_grad()
def olc(yol):
    """Checkpoint'i gerçek arşivdeki gibi ölçer: yuvarlanmış notlar + checkpoint'in bit sayısında ressam (gerçek format)."""
    ck = torch.load(yol, map_location=device)
    bits = ck.get("bits", 8)
    m = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    m.load_state_dict(ck["model"])
    kb = sum(entropy_kb(torch.round(m.grids[k][PILOT])) for k in m.grids) * NUM_PAIRS / len(PILOT)
    r_kb = ressam_kb_bit(ck["model"], bits)
    quant_net_(m, bits)
    d8 = score(m, PILOT)["dist"]
    del m; torch.cuda.empty_cache()
    return d8, kb, d8 + rate_of_kb(kb + r_kb)


@torch.no_grad()
def genislet(eski_yol, yeni_widths, tag):
    """Ressamı işlevi koruyarak genişletir (Net2Net tarzı).
    Eski kanallar aynen kopyalanır. Yeni kanalların GİRİŞ ağırlıkları rastgele kalır (öğrenebilsinler),
    eski çıkışlara giden ağırlıkları 0 yapılır → başlangıçta çıktı birebir aynı."""
    ck = torch.load(eski_yol, map_location=device)
    eski = GridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    eski.load_state_dict(ck["model"])
    torch.manual_seed(0)
    yeni = GridDecoder(NUM_PAIRS, GRID_LEVELS, yeni_widths).to(device)
    for k in eski.grids:
        yeni.grids[k].copy_(eski.grids[k])

    def kopyala(e, y, n_out, n_in):
        y.weight[:n_out, n_in:] = 0                  # yeni kanallardan eski çıkışlara: 0
        y.weight[:n_out, :n_in] = e.weight           # eski → eski: aynen
        y.bias[:n_out] = e.bias

    for d in eski.inject:                            # notlar → ressam (giriş kanalı sayısı değişmiyor)
        n = eski.inject[d].weight.shape[0]
        yeni.inject[d].weight[:n] = eski.inject[d].weight
        yeni.inject[d].bias[:n] = eski.inject[d].bias
    for i in range(6):
        kopyala(eski.ups[i], yeni.ups[i], 4 * WIDTHS[i + 1], WIDTHS[i])   # pixel_shuffle: kanal c → satır 4c..4c+3
        kopyala(eski.skips[i], yeni.skips[i], WIDTHS[i + 1], WIDTHS[i])
    kopyala(eski.head, yeni.head, 6, WIDTHS[6])

    ids = PILOT[:8]                                  # sağlama: çıktı aynı mı?
    fark = (eski(ids) - yeni(ids)).abs().max().item()
    n_eski = sum(p.numel() for n, p in eski.named_parameters() if not n.startswith("grids"))
    n_yeni = sum(p.numel() for n, p in yeni.named_parameters() if not n.startswith("grids"))
    print(f"[{tag}] genişletme: {n_eski:,} → {n_yeni:,} parametre | çıktı farkı {fark:.2e} (≈0 olmalı)")
    torch.save({"model": yeni.state_dict(), "log_b": ck["log_b"], "step0": ck.get("step0"), "ent": ck.get("ent")},
               OUT_DIR / f"pilot_rate_{tag}.pt")
    del eski, yeni; torch.cuda.empty_cache()


def devam_turlari(on_ek, ilk=1, son=8, esik=0.005, bit_ep=3000, ft_ep=600, ft_eval=100,
                  parca=None, birikim=1):
    """Devam turları. Bit aşaması: `parca` × `birikim` etkin batch (LR etkin batch'e göre ölçeklenir)."""
    parca = parca or B_BIT
    sonuclar = json.loads(SONUC.read_text()) if SONUC.exists() else {}
    onceki = OUT_DIR / (f"pilot_rate_{on_ek}_ft{ft_ep}.pt" if ilk == 1
                        else f"pilot_rate_{on_ek}_devam{ilk - 1}_ft{ft_ep}.pt")
    with open(LOG, "a") as f, redirect_stdout(Tee(sys.stdout, f)):
        print(f"\n=== {on_ek} devam turları başladı {time.strftime('%Y-%m-%d %H:%M')} | {len(PILOT)} pair ===")
        _, _, t_onceki = olc(onceki)
        print(f"[başlangıç] {onceki.name}: toplam {t_onceki:.3f}")
        for n in range(ilk, son + 1):
            T = f"{on_ek}_devam{n}"
            bit_yol = OUT_DIR / f"pilot_rate_{T}.pt"
            ft_yol = OUT_DIR / f"pilot_rate_{T}_ft{ft_ep}.pt"
            t0 = time.time()
            if ft_yol.exists():
                print(f"[{T}] zaten bitmiş, ölçülüp geçiliyor")
            else:
                if not bit_yol.exists():
                    s = (parca * birikim / 8) ** 0.5                   # LR etkin batch'e göre
                    run_rate(30, bit_ep, onceki, T, ramp_ep=1, lam_min=30, eval_every=250, batch=parca,
                             birikim=birikim, lr_carpan=0.3 * s, grid_carpan=0.55 / s, b_min=0.3)
                B = B_FT; s = (B / 8) ** 0.5
                net_ft(bit_yol, f"{T}_ft{ft_ep}", epochs=ft_ep, batch=B, lr=3e-4 * s, eval_every=ft_eval)
            d8, kb, t = olc(ft_yol)
            kazanc = t_onceki - t
            sonuclar[T] = {"ayar": f"devam{n}", "yontem": "sert", "lam": 30, "pairs": len(PILOT),
                           "dist": d8, "dist8": d8, "kb": kb, "net_kb": NET_KB,
                           "fp32_dist": sonuclar.get(f"_fp32_{len(PILOT)}", 0.0835), "toplam": t,
                           "dk": round((time.time() - t0) / 60)}
            SONUC.write_text(json.dumps(sonuclar, indent=1))
            print(f"[{T}] TUR SONU | 8 bit {d8:.4f} | notlar {kb:.0f} KB | toplam {t:.3f} | kazanç {kazanc:+.3f}")
            if kazanc < esik:
                print(f"[{T}] kazanç {esik} altında → turlar durdu. En iyi: {onceki.name if kazanc < 0 else ft_yol.name}")
                break
            onceki, t_onceki = ft_yol, t
            gc.collect(); torch.cuda.empty_cache()
        ozet(sonuclar)


def net_ft_q(init_path, tag, bits=8, sinif=None, kod_lr=1e-2, epochs=600, batch=B_FT,
             lr=3e-4 * (B_FT / 8) ** 0.5, warm=10, eval_every=100, zor_beta=0.0, zor_yenile=50):
    """net_ft + QAT, `bits` bitlik ressamla. Eğitim yuvarlanmış ağırlıkla ileri/geri, adım FP32 ağırlığa (STE).
    En iyi checkpoint `bits` bitlik dist'e göre; ressam KB'ı gerçek formatla (kanal başına entropi + fp16 bias/ölçek)."""
    ck = torch.load(init_path, map_location=device)
    sinif = sinif or GridDecoder
    m = sinif(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    m.load_state_dict(ck["model"], strict=sinif is GridDecoder)   # ModGridDecoder: codes/film checkpoint'te yoksa yeni başlar
    with torch.no_grad():
        for g in m.grids.values(): g.round_()
    m.grids.requires_grad_(False)
    kb = sum(entropy_kb(m.grids[k][PILOT]) for k in m.grids)
    params = [p for n, p in m.named_parameters() if not n.startswith("grids")]
    ressam_p = [p for n, p in m.named_parameters() if not n.startswith(("grids", "codes"))]
    kod_p = [p for n, p in m.named_parameters() if n.startswith("codes")]
    gruplar = [{"params": ressam_p, "lr": lr}]
    if kod_p:
        gruplar.append({"params": kod_p, "lr": kod_lr})   # kodlar notlar gibi pair başına: ressam LR'siyle yerinden oynamıyor
    adim = math.ceil(len(PILOT) / batch)
    opt = torch.optim.Adam(gruplar)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / (warm * adim))
                                              * 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    print(f"[{tag}] net_ft_q | QAT {bits} bit | batch {batch} | LR {lr:.2e} | notlar {kb:.1f} KB "
          f"| ressam başta {ressam_kb_bit(ck['model'], bits):.1f} KB")
    en_iyi = {"dq": float("inf")}
    agirlik = None
    for ep in tqdm(range(epochs)):
        if zor_beta and ep % zor_yenile == 0:
            z = pair_zorluk(m)                                           # bkz. bölüm 14
            agirlik = ((z / z.mean()) ** zor_beta).clamp(0.25, 4.0)      # uçlar kırpılır ki kapsama kaybolmasın
        perm = (PILOT[torch.multinomial(agirlik, len(PILOT), replacement=True)] if agirlik is not None
                else PILOT[torch.randperm(len(PILOT), device=device)])
        for i in range(0, len(perm), batch):
            ids = perm[i:i + batch]
            asil = qat_ac(m, bits)
            loss, _, _ = task_loss(m(ids), ids)
            opt.zero_grad(set_to_none=True); loss.backward()
            qat_kapat(m, asil)
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step()
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            em = copy.deepcopy(m)
            sd = copy.deepcopy(em.state_dict())
            r_kb = ressam_kb_bit(sd, bits)
            with torch.no_grad():
                quant_net_(em, bits)
            rq = score(em, PILOT)
            top = rq["dist"] + rate_of_kb(kb + r_kb)
            if top < en_iyi.get("top", float("inf")):
                en_iyi = {"dq": rq["dist"], "top": top, "rq": rq, "ep": ep + 1, "sd": sd, "r_kb": r_kb}
            del em
            print(f"[{tag}] E{ep + 1:03d} | {bits} bit dist {rq['dist']:.4f} (seg {rq['seg']:.4f}) "
                  f"| ressam {r_kb:.1f} KB | toplam {top:.3f} | en iyi {en_iyi['top']:.3f} (E{en_iyi['ep']})")
    rq = en_iyi["rq"]
    print(f"[{tag}] SON | en iyi E{en_iyi['ep']} | {bits} bit dist {rq['dist']:.4f} (seg {rq['seg']:.4f}, "
          f"pose {rq['pose']:.4f}) | notlar {kb:.1f} + ressam {en_iyi['r_kb']:.1f} KB | TOPLAM {en_iyi['top']:.3f}")
    torch.save({"model": en_iyi["sd"], "log_b": ck["log_b"], "step0": ck.get("step0"), "ent": ck.get("ent"),
                "bits": bits}, OUT_DIR / f"pilot_rate_{tag}.pt")
    del m; torch.cuda.empty_cache()


## 10. Pair başına modülasyon (FiLM)

In [ ]:
class ModGridDecoder(GridDecoder):
    """Pair başına kod → her kademede kanal modülasyonu: x ← x·(1+γ) + β (FiLM / COIN++)."""
    def __init__(self, n_pairs, levels=GRID_LEVELS, widths=WIDTHS, k=16):
        super().__init__(n_pairs, levels, widths)
        self.widths = list(widths)
        self.codes = nn.Parameter(0.1 * torch.randn(n_pairs, k))           # ikisi de 0 olursa gradyan ölür
        self.film = nn.Linear(k, 2 * sum(widths[1:]))
        nn.init.zeros_(self.film.weight); nn.init.zeros_(self.film.bias)  # başta birim dönüşüm → lr15 çıktısı aynen

    def forward(self, ids, grids=None):
        g = grids if grids is not None else {k: v[ids] for k, v in self.grids.items()}
        gb = self.film(self.codes[ids]).split([w for w in self.widths[1:] for _ in (0, 1)], dim=1)
        x = self.inject["64"](g["64"])
        for i in range(6):
            up = F.pixel_shuffle(self.ups[i](x), 2)
            skip = F.interpolate(self.skips[i](x), scale_factor=2, mode="bilinear", align_corners=False)
            x = F.gelu(up + skip)
            x = x * (1 + gb[2 * i][:, :, None, None]) + gb[2 * i + 1][:, :, None, None]
            d = str(64 // 2 ** (i + 1))
            if d in self.inject:
                x = x + self.inject[d](g[d])
        return (torch.sigmoid(self.head(x)) * 255).view(len(ids), 2, 3, 384, 512)


## 11. Deneyler
Her hücre `PILOT` + `GRID_LEVELS` / `WIDTHS` ile başlar. Bitince `SON` satırını not al.

In [ ]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [20, 20, 20, 20, 16, 16, 10]
mimari_kur(GRID_LEVELS, WIDTHS, 600)

# Kapı deneyi: notlar donuk, kodlar fp32, kontrol = lr15 (dist 0.2325, aynı başlangıç ve LR).
# ressam_kb_bit kodları ve film'i fp16 sayar (600×16 kod ≈ 19 KB); karar dist kazancına göre (hedef ≥ 0.012).
net_ft_q(OUT_DIR / "pilot_rate_p600u_genis_qat8_lr15.pt", "p600u_film16", bits=8, lr=1.5e-4, sinif=ModGridDecoder)


[p600u_film16] net_ft_q | QAT 8 bit | batch 40 | LR 1.50e-04 | notlar 51.9 KB | ressam başta 67.4 KB


  0%|          | 0/600 [00:00<?, ?it/s]

[p600u_film16] E100 | 8 bit dist 0.2600 (seg 0.2100) | ressam 93.5 KB | toplam 0.357 | en iyi 0.357 (E100)
[p600u_film16] E200 | 8 bit dist 0.2597 (seg 0.2073) | ressam 93.5 KB | toplam 0.357 | en iyi 0.357 (E200)
[p600u_film16] E300 | 8 bit dist 0.2527 (seg 0.2059) | ressam 93.5 KB | toplam 0.349 | en iyi 0.349 (E300)
[p600u_film16] E400 | 8 bit dist 0.2385 (seg 0.2017) | ressam 93.5 KB | toplam 0.335 | en iyi 0.335 (E400)
[p600u_film16] E500 | 8 bit dist 0.2374 (seg 0.2034) | ressam 93.5 KB | toplam 0.334 | en iyi 0.334 (E500)
[p600u_film16] E600 | 8 bit dist 0.2266 (seg 0.1973) | ressam 93.5 KB | toplam 0.323 | en iyi 0.323 (E600)
[p600u_film16] SON | en iyi E600 | 8 bit dist 0.2266 (seg 0.1973, pose 0.0293) | notlar 51.9 + ressam 93.5 KB | TOPLAM 0.323


In [ ]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [20, 20, 20, 20, 16, 16, 10]
mimari_kur(GRID_LEVELS, WIDTHS, 600)

# Hazır kalıplar (çalıştırmadan önce etiketi değiştir):
# genislet(OUT_DIR / "pilot_rate_X.pt", [16, 16, 16, 16, 16, 16, 12], "X_genis")
# bit_tarama("p600u_genis_qat8_lr15", WIDTHS)
# devam_turlari(on_ek="p600u_genis", ilk=1, son=3, parca=34, birikim=3)   # ft aşaması henüz net_ft (QAT'sız)
print(olc(OUT_DIR / "pilot_rate_p600u_genis_qat8_lr15.pt"))


(0.23251459304352404, 51.8706679958901, 0.3119161061955752)


In [ ]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [20, 20, 20, 20, 16, 16, 10]
mimari_kur(GRID_LEVELS, WIDTHS, 600)

net_ft_q(OUT_DIR / "pilot_rate_p600u_genis_qat8_lr15.pt", "p600u_film8_kodlr", bits=8, lr=1.5e-4,
         sinif=lambda n, l, w: ModGridDecoder(n, l, w, k=8), kod_lr=1e-2)

[p600u_film8_kodlr] net_ft_q | QAT 8 bit | batch 40 | LR 1.50e-04 | notlar 51.9 KB | ressam başta 67.4 KB


  0%|          | 0/600 [00:00<?, ?it/s]

[p600u_film8_kodlr] E100 | 8 bit dist 0.2500 (seg 0.2062) | ressam 80.7 KB | toplam 0.338 | en iyi 0.338 (E100)
[p600u_film8_kodlr] E200 | 8 bit dist 0.2535 (seg 0.2056) | ressam 80.7 KB | toplam 0.342 | en iyi 0.338 (E100)
[p600u_film8_kodlr] E300 | 8 bit dist 0.2439 (seg 0.2062) | ressam 80.7 KB | toplam 0.332 | en iyi 0.332 (E300)
[p600u_film8_kodlr] E400 | 8 bit dist 0.2396 (seg 0.2023) | ressam 80.7 KB | toplam 0.328 | en iyi 0.328 (E400)
[p600u_film8_kodlr] E500 | 8 bit dist 0.2362 (seg 0.2010) | ressam 80.7 KB | toplam 0.324 | en iyi 0.324 (E500)
[p600u_film8_kodlr] E600 | 8 bit dist 0.2245 (seg 0.1972) | ressam 80.7 KB | toplam 0.313 | en iyi 0.313 (E600)
[p600u_film8_kodlr] SON | en iyi E600 | 8 bit dist 0.2245 (seg 0.1972, pose 0.0273) | notlar 51.9 + ressam 80.7 KB | TOPLAM 0.313


In [ ]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [20, 20, 20, 20, 16, 16, 10]
mimari_kur(GRID_LEVELS, WIDTHS, 600)


def q8_kb(t, bits=8):
    """Satır başına simetrik nicemleme (kodlarda satır = boyut, film'de satır = çıkış). Döner: yuvarlanmış tensör, KB."""
    qmax = 2 ** (bits - 1) - 1
    s = t.abs().amax(dim=1, keepdim=True).clamp_min(1e-12) / qmax
    kod = torch.round(t / s).clamp(-qmax, qmax)
    p = torch.bincount((kod + qmax).long().flatten()).float(); p = p[p > 0] / kod.numel()
    bit = -(p * p.log2()).sum().item() * kod.numel() + 16 * t.shape[0]
    return kod * s, bit / 8 / 1000


@torch.no_grad()
def olc_film(yol, k=8, bits=8):
    ck = torch.load(yol, map_location=device)
    m = ModGridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS, k=k).to(device)
    m.load_state_dict(ck["model"])
    for g in m.grids.values(): g.round_()
    not_kb = sum(entropy_kb(m.grids[kk][PILOT]) for kk in m.grids)
    sd = {n: v for n, v in ck["model"].items() if not n.startswith(("codes", "film"))}
    r_kb = ressam_kb_bit(sd, bits)
    d_fp = score(m, PILOT)["dist"]
    quant_net_(m, bits)
    d_conv = score(m, PILOT)["dist"]
    # kodlar boyut başına ölçeklenir: her boyutun aralığı farklı, pair'ler arası aynı
    q, kod_kb = q8_kb(m.codes.data.t(), bits); m.codes.data = q.t()
    q, film_kb = q8_kb(m.film.weight.data, bits); m.film.weight.data = q
    film_kb += 16 * m.film.bias.numel() / 8 / 1000
    d_tam = score(m, PILOT)["dist"]
    kb = not_kb + r_kb + kod_kb + film_kb
    print(f"dist fp {d_fp:.4f} | conv 8 bit {d_conv:.4f} | + kod/film 8 bit {d_tam:.4f} (seg ve pose ayrı değil)")
    print(f"notlar {not_kb:.1f} + ressam {r_kb:.1f} + kodlar {kod_kb:.1f} + film {film_kb:.1f} = {kb:.1f} KB "
          f"| TOPLAM {d_tam + rate_of_kb(kb):.3f}")


olc_film(OUT_DIR / "pilot_rate_p600u_film8_kodlr.pt")

dist fp 0.2294 | conv 8 bit 0.2245 | + kod/film 8 bit 0.2245 (seg ve pose ayrı değil)
notlar 51.9 + ressam 67.4 + kodlar 4.3 + film 2.3 = 125.9 KB | TOPLAM 0.308


In [ ]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [20, 20, 20, 20, 16, 16, 10]
mimari_kur(GRID_LEVELS, WIDTHS, 600)

FILM8 = lambda n, l, w: ModGridDecoder(n, l, w, k=8)
s = (34 * 3 / 8) ** 0.5
run_rate(30, 3000, OUT_DIR / "pilot_rate_p600u_film8_kodlr.pt", "p600u_film8_devam1",
         ramp_ep=1, lam_min=30, eval_every=250, batch=34, birikim=3,
         lr_carpan=0.3 * s, grid_carpan=0.55 / s, b_min=0.3, sinif=FILM8)
net_ft_q(OUT_DIR / "pilot_rate_p600u_film8_devam1.pt", "p600u_film8_devam1_ft600", bits=8,
         sinif=FILM8, kod_lr=1e-2)
olc_film(OUT_DIR / "pilot_rate_p600u_film8_devam1_ft600.pt")

[p600u_film8_devam1] bit aşaması | parça 34 × birikim 3 = etkin batch 102 | 6 adım/epoch | not LR 4.95e-03 | ressam LR 3.21e-04


  0%|          | 0/3000 [00:00<?, ?it/s]

[p600u_film8_devam1] E250 λ=30 | dist 0.3154 | ızgara ~52 KB | sıfır %98/%96/%35 | toplam ≈ 0.399
[p600u_film8_devam1] E500 λ=30 | dist 0.3365 | ızgara ~52 KB | sıfır %98/%96/%34 | toplam ≈ 0.420
[p600u_film8_devam1] E750 λ=30 | dist 0.3393 | ızgara ~51 KB | sıfır %98/%96/%34 | toplam ≈ 0.422
[p600u_film8_devam1] E1000 λ=30 | dist 0.3257 | ızgara ~50 KB | sıfır %98/%97/%33 | toplam ≈ 0.408
[p600u_film8_devam1] E1250 λ=30 | dist 0.3304 | ızgara ~49 KB | sıfır %98/%97/%34 | toplam ≈ 0.412
[p600u_film8_devam1] E1500 λ=30 | dist 0.3227 | ızgara ~49 KB | sıfır %98/%97/%34 | toplam ≈ 0.404
[p600u_film8_devam1] E1750 λ=30 | dist 0.3124 | ızgara ~49 KB | sıfır %98/%97/%33 | toplam ≈ 0.393
[p600u_film8_devam1] E2000 λ=30 | dist 0.3086 | ızgara ~48 KB | sıfır %98/%97/%33 | toplam ≈ 0.389
[p600u_film8_devam1] E2250 λ=30 | dist 0.3087 | ızgara ~48 KB | sıfır %98/%97/%33 | toplam ≈ 0.389
[p600u_film8_devam1] E2500 λ=30 | dist 0.3110 | ızgara ~48 KB | sıfır %98/%97/%33 | toplam ≈ 0.391
[p600u_film8_

  0%|          | 0/600 [00:00<?, ?it/s]

[p600u_film8_devam1_ft600] E100 | 8 bit dist 0.2914 (seg 0.2153) | ressam 81.0 KB | toplam 0.376 | en iyi 0.376 (E100)
[p600u_film8_devam1_ft600] E200 | 8 bit dist 0.3047 (seg 0.2086) | ressam 81.0 KB | toplam 0.389 | en iyi 0.376 (E100)
[p600u_film8_devam1_ft600] E300 | 8 bit dist 0.2676 (seg 0.2056) | ressam 81.0 KB | toplam 0.352 | en iyi 0.352 (E300)
[p600u_film8_devam1_ft600] E400 | 8 bit dist 0.2437 (seg 0.2012) | ressam 80.9 KB | toplam 0.328 | en iyi 0.328 (E400)
[p600u_film8_devam1_ft600] E500 | 8 bit dist 0.2369 (seg 0.1953) | ressam 80.9 KB | toplam 0.321 | en iyi 0.321 (E500)
[p600u_film8_devam1_ft600] E600 | 8 bit dist 0.2149 (seg 0.1904) | ressam 80.9 KB | toplam 0.299 | en iyi 0.299 (E600)
[p600u_film8_devam1_ft600] SON | en iyi E600 | 8 bit dist 0.2149 (seg 0.1904, pose 0.0245) | notlar 45.4 + ressam 80.9 KB | TOPLAM 0.299
dist fp 0.2195 | conv 8 bit 0.2149 | + kod/film 8 bit 0.2150 (seg ve pose ayrı değil)
notlar 45.4 + ressam 67.7 + kodlar 4.3 + film 2.3 = 119.7 KB | 

In [ ]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [20, 20, 20, 20, 16, 16, 10]
mimari_kur(GRID_LEVELS, WIDTHS, 600)

FILM8 = lambda n, l, w: ModGridDecoder(n, l, w, k=8)
s = (34 * 3 / 8) ** 0.5
run_rate(30, 3000, OUT_DIR / "pilot_rate_p600u_film8_devam1_ft600.pt", "p600u_film8_devam2",
         ramp_ep=1, lam_min=30, eval_every=250, batch=34, birikim=3,
         lr_carpan=0.3 * s, grid_carpan=0.55 / s, b_min=0.3, sinif=FILM8)
net_ft_q(OUT_DIR / "pilot_rate_p600u_film8_devam2.pt", "p600u_film8_devam2_ft600", bits=8,
         sinif=FILM8, kod_lr=1e-2)
olc_film(OUT_DIR / "pilot_rate_p600u_film8_devam2_ft600.pt")

[p600u_film8_devam2] bit aşaması | parça 34 × birikim 3 = etkin batch 102 | 6 adım/epoch | not LR 4.95e-03 | ressam LR 3.21e-04


  0%|          | 0/3000 [00:00<?, ?it/s]

## 12. (v6) Zamansal entropi + delta kod — kayıpsız
600 pair ardışık video karesi; notlar, kodlar ve entropi modeli şimdiye dek her pair'i bağımsız saydı.
Buradaki her şey **kayıpsız**: dist'e dokunmaz, yalnız kod uzunluğunu düşürür. Entropi modeli donmuş,
yuvarlanmış notlar üzerinde **sonradan** eğitilir (böl. 6 notu: birlikte eğitim notları kötüleştiriyordu).
Çözüm sırası: pair 0→599, seviye 64→32→16, seviye içinde raster — önceki pair tamamen çözülmüş olduğundan
`prev` dalında nedensel maske gerekmez.

In [ ]:
class TemporalLevelEntropy(nn.Module):
  """LevelEntropy + bir önceki pair'in aynı seviye notları. Önceki pair çözülmüş olduğundan maske
  gerekmez; 5×5 pencere kameranın iki pair arası kaymasını da yakalasın diye."""
  def __init__(self, c=2, h=16, has_coarse=True):
    super().__init__()
    self.same = MaskedConv2d(c, h, 5)
    self.prev = nn.Conv2d(c, h, 5, padding=2)
    self.coarse = nn.Conv2d(c, h, 1) if has_coarse else None
    self.mix = nn.Sequential(
        nn.GELU(), nn.Conv2d(h, h, 1),
        nn.GELU(), nn.Conv2d(h, 2 * c, 1),
    )
    nn.init.zeros_(self.mix[-1].weight)
    nn.init.zeros_(self.mix[-1].bias)
  def forward(self, q, q_coarse, q_prev):
    x = self.same(q) + self.prev(q_prev)
    if self.coarse is not None:
      x = x + self.coarse(F.interpolate(q_coarse, size=q.shape[-2:], mode="nearest"))
    mu, log_s = self.mix(x).chunk(2, dim=1)
    return mu, log_s.exp().clamp(0.1)


class TemporalContextEntropy(nn.Module):
  ORDER = ["64", "32", "16"]
  def __init__(self, levels, h=16):
    super().__init__()
    self.nets = nn.ModuleDict({k: TemporalLevelEntropy(levels[int(k)], h=h, has_coarse=(i > 0))
                               for i, k in enumerate(self.ORDER)})
  def forward(self, q, q_prev):
    out = {}
    for i, k in enumerate(self.ORDER):
      onceki = q[self.ORDER[i - 1]] if i > 0 else None
      out[k] = self.nets[k](q[k], onceki, q_prev[k])
    return out


def onceki_q(grids, ids):
    """ids-1'in notları; pair 0 videonun başı olduğundan bağlamı sıfır."""
    prev = (ids - 1).clamp_min(0)
    mask = (ids > 0).view(-1, 1, 1, 1).float()
    return {k: g[prev] * mask for k, g in grids.items()}

In [ ]:
@torch.no_grad()
def temporal_kb(ent, grids, bs=50):
    """Tamsayı notların zamansal model altındaki kod uzunluğu, seviye seviye KB (tüm pair'ler)."""
    toplam = {k: 0.0 for k in TemporalContextEntropy.ORDER}
    for s0 in range(0, NUM_PAIRS, bs):
        ids = torch.arange(s0, min(s0 + bs, NUM_PAIRS), device=device)
        q = {k: torch.round(g[ids]) for k, g in grids.items()}
        qp = {k: torch.round(v) for k, v in onceki_q(grids, ids).items()}
        for k, (mu, sc) in ent(q, qp).items():
            toplam[k] += laplace_bits(q[k] - mu, sc).sum().item()
    return {k: v / 8 / 1e3 for k, v in toplam.items()}


def ent_egit(ck_yol, tag, h=16, epochs=200, lr=3e-3, bs=50):
    """Zamansal entropi modelini checkpoint'in DONMUŞ tamsayı notları üzerinde eğitir (saf yoğunluk
    tahmini, ressama dokunmaz). Tamsayı + ±0.5 kutu olasılığı ayrık MLE'ye denk geldiğinden gürültü
    gerekmez. Kazanç = iid − (zamansal + ent ağı); pozitifse doğrudan skor düşer."""
    ck = torch.load(ck_yol, map_location="cpu")
    grids = {str(d): torch.round(ck["model"][f"grids.{d}"]).to(device) for d in GRID_LEVELS}
    ent = TemporalContextEntropy(GRID_LEVELS, h=h).to(device)
    opt = torch.optim.Adam(ent.parameters(), lr=lr)
    adim = math.ceil(NUM_PAIRS / bs)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 0.5 * (1 + math.cos(math.pi * min(1, s / (epochs * adim)))))
    iid = {k: entropy_kb(v) for k, v in grids.items()}
    for ep in range(epochs):
        perm = torch.randperm(NUM_PAIRS, device=device)
        for i in range(0, NUM_PAIRS, bs):
            ids = perm[i:i + bs]
            q = {k: g[ids] for k, g in grids.items()}
            bits = sum(laplace_bits(q[k] - mu, sc).sum() for k, (mu, sc) in ent(q, onceki_q(grids, ids)).items())
            opt.zero_grad(set_to_none=True); (bits / len(ids)).backward(); opt.step(); sched.step()
        if (ep + 1) % 50 == 0 or ep + 1 == epochs:
            t = temporal_kb(ent, grids)
            kazanc = sum(iid.values()) - sum(t.values()) - ent_kb(ent)
            satir = " | ".join(f"1/{k} {iid[k]:.1f}>{t[k]:.1f}" for k in t)
            print(f"[{tag}] E{ep + 1:03d} | {satir} KB | ent {ent_kb(ent):.1f} KB "
                  f"| kazanç {kazanc:+.1f} KB ({-kazanc * rate_of_kb(1):+.4f} skor)")
    torch.save({"ent": ent.state_dict(), "h": h}, OUT_DIR / f"ent_temp_{tag}.pt")
    return ent

In [ ]:
def q8_delta_kb(t, bits=8):
    """q8_kb ile aynı nicemleme; bit, zaman ekseni (sütunlar) boyunca farklardan ölçülür.
    Çözücü aynı tamsayıları toplayarak geri alır → kayıpsız, dist değişmez."""
    qmax = 2 ** (bits - 1) - 1
    s = t.abs().amax(dim=1, keepdim=True).clamp_min(1e-12) / qmax
    kod = torch.round(t / s).clamp(-qmax, qmax)
    fark = torch.cat([kod[:, :1], kod[:, 1:] - kod[:, :-1]], dim=1)
    return kod * s, entropy_kb(fark) + 16 * t.shape[0] / 8 / 1000


@torch.no_grad()
def olc_film2(yol, k=8, bits=8, ent=None):
    """olc_film + kayıpsız v6 kazançları: notlar zamansal entropiyle (verilirse; modül ya da
    ent_temp_*.pt yolu), kodlar delta kodla. Kalem kalem iid ile karşılaştırır, ucuzunu toplar."""
    ck = torch.load(yol, map_location=device)
    m = ModGridDecoder(NUM_PAIRS, GRID_LEVELS, WIDTHS, k=k).to(device)
    m.load_state_dict(ck["model"])
    for g in m.grids.values(): g.round_()
    not_iid = sum(entropy_kb(m.grids[kk][PILOT]) for kk in m.grids)
    not_kb, not_ek = not_iid, ""
    if ent is not None:
        if not isinstance(ent, nn.Module):
            e_ck = torch.load(ent, map_location=device)
            e = TemporalContextEntropy(GRID_LEVELS, h=e_ck["h"]).to(device)
            e.load_state_dict(e_ck["ent"]); ent = e
        not_tmp = sum(temporal_kb(ent, dict(m.grids.items())).values()) + ent_kb(ent)
        not_kb, not_ek = min(not_iid, not_tmp), f" (iid {not_iid:.1f} / zamansal {not_tmp:.1f})"
    sd = {n: v for n, v in ck["model"].items() if not n.startswith(("codes", "film"))}
    r_kb = ressam_kb_bit(sd, bits)
    quant_net_(m, bits)
    q, kod_iid = q8_kb(m.codes.data.t(), bits)
    _, kod_dlt = q8_delta_kb(m.codes.data.t(), bits)                 # aynı tamsayılar → dist aynı
    m.codes.data = q.t()
    kod_kb = min(kod_iid, kod_dlt)
    q, film_kb = q8_kb(m.film.weight.data, bits); m.film.weight.data = q
    film_kb += 16 * m.film.bias.numel() / 8 / 1000
    d_tam = score(m, PILOT)["dist"]
    kb = not_kb + r_kb + kod_kb + film_kb
    print(f"dist ({bits} bit ressam + kod/film) {d_tam:.4f}")
    print(f"notlar {not_kb:.1f}{not_ek} + ressam {r_kb:.1f} + kodlar {kod_kb:.1f} "
          f"(iid {kod_iid:.1f} / delta {kod_dlt:.1f}) + film {film_kb:.1f} = {kb:.1f} KB "
          f"| TOPLAM {d_tam + rate_of_kb(kb):.3f}")
    del m; torch.cuda.empty_cache()

## 13. (v6) Seviye raporu ve kanal cerrahisi
Loglardaki sıfır oranları (%98/%97/%34) kaba seviyelerin neredeyse boş taşındığını söylüyor.
`seviye_raporu` her seviye/kanalın KB maliyetiyle dist katkısını karşılaştırır (kural: 1 KB ≈ 0.00067 skor);
kârsız kanal `kanal_sil` + bir devam turuyla atılır. Sıfırdan mimari varyantları 120 pair gece kuyruğunda (böl. 15).

In [ ]:
@torch.no_grad()
def seviye_raporu(yol, sinif=None, kanal=True):
    """Seviye/kanal maliyet-katkı dengesi: sıfırlanınca dist artışı < KB'nin skor karşılığı ise kârsız.
    Her satır tam bir skor ölçümü; kanal=True ile ~9 ölçüm sürer."""
    ck = torch.load(yol, map_location=device)
    m = (sinif or GridDecoder)(NUM_PAIRS, GRID_LEVELS, WIDTHS).to(device)
    m.load_state_dict(ck["model"])
    for g in m.grids.values(): g.round_()
    quant_net_(m, ck.get("bits", 8))
    d0 = score(m, PILOT)["dist"]
    print(f"taban dist {d0:.4f} | karar: Δdist < KB karşılığı → kârsız")
    for k, g in m.grids.items():
        kb = entropy_kb(g[PILOT]) * NUM_PAIRS / len(PILOT)
        yedek = g.data.clone(); g.data.zero_()
        d = score(m, PILOT)["dist"]; g.data.copy_(yedek)
        esik = rate_of_kb(kb)
        etiket = "← KÂRSIZ" if d - d0 < esik else ""
        print(f"1/{k}: {kb:5.1f} KB (skor {esik:.4f}) | sıfır %{100 * (g[PILOT] == 0).float().mean():.0f} "
              f"| sıfırlanırsa Δdist +{d - d0:.4f} {etiket}")
        if kanal and g.shape[1] > 1:
            for c in range(g.shape[1]):
                kbc = entropy_kb(g[PILOT][:, c]) * NUM_PAIRS / len(PILOT)
                yc = g.data[:, c].clone(); g.data[:, c] = 0
                dc = score(m, PILOT)["dist"]; g.data[:, c] = yc
                ec = rate_of_kb(kbc)
                etiket = "← KÂRSIZ" if dc - d0 < ec else ""
                print(f"   kanal {c}: {kbc:5.1f} KB (skor {ec:.4f}) | Δdist +{dc - d0:.4f} {etiket}")
    del m; torch.cuda.empty_cache()


@torch.no_grad()
def kanal_sil(eski_yol, tut, tag, sinif=None):
    """tut: {64: [0], 32: [0], 16: [0, 1]} — TÜM seviyeler için tutulacak kanallar
    (64'ten en az 1 kalmalı: decoder oradan başlar; film checkpoint'i için sinif ver).
    Atılan kanalların sıfır olmayan ~%2'si kaybolur → ardından mimari_kur(yeni levels) + devam turu şart."""
    assert len(tut[64]) >= 1
    ck = torch.load(eski_yol, map_location=device)
    yeni_levels = {d: len(tut[d]) for d in GRID_LEVELS}
    sd = dict(ck["model"])
    log_b = {}
    for d, kanallar in tut.items():
        idx = torch.tensor(kanallar, device=device)
        sd[f"grids.{d}"] = sd[f"grids.{d}"][:, idx]
        sd[f"inject.{d}.weight"] = sd[f"inject.{d}.weight"][:, idx]
        log_b[str(d)] = ck["log_b"][str(d)][idx]
    m = (sinif or GridDecoder)(NUM_PAIRS, yeni_levels, WIDTHS).to(device)
    m.load_state_dict(sd)                                            # şekil sağlaması
    torch.save({"model": m.state_dict(), "log_b": log_b, "step0": ck.get("step0"), "ent": None,
                "bits": ck.get("bits", 8)}, OUT_DIR / f"pilot_rate_{tag}.pt")
    print(f"[{tag}] {dict(GRID_LEVELS)} → {yeni_levels} | sıradaki: mimari_kur({yeni_levels}, WIDTHS, 600) + devam turu")
    del m; torch.cuda.empty_cache()

## 14. (v6) Seg odaklı: l7 ayarları, sınıf ağırlığı, zor-pair örnekleme
dist'in ~%89'u seg. `L7` globali l7_loss ayarlarını taşır (tarama için), `SINIF_W` nadir sınıflara
(%0.6 ve %1.2) ağırlık verir. `net_ft_q(zor_beta=...)` kötü pair'leri daha sık örnekler; en-iyi seçimi
yine tüm PILOT skoruyla yapıldığından ezbere kayma riski ölçüme yansır.

In [ ]:
L7 = dict(tau=0.3, thr=1.0, mult=4.0)                  # v5 varsayılanları
SINIF_W = None                                          # örn. torch.tensor([1., 4., 1., 3., 1.], device=device)

def l7_loss(logits, truth, tau=None, thr=None, mult=None):
  """Böl. 4'tekiyle aynı; ayarlar L7 globalinden gelir ki tarama fonksiyon imzalarına dokunmasın."""
  tau = L7["tau"] if tau is None else tau
  thr = L7["thr"] if thr is None else thr
  mult = L7["mult"] if mult is None else mult
  correct = logits.gather(1, truth.unsqueeze(1))
  others = logits.clone()
  others.scatter_(1, truth.unsqueeze(1), torch.finfo(others.dtype).min)
  margin = correct - others.amax(1, keepdim=True)
  per_pixel = tau * F.softplus(-margin / tau)
  with torch.no_grad():
    w = 1.0 + mult * (margin < thr).float()
    if SINIF_W is not None:
      w = w * SINIF_W.to(margin.device)[truth].unsqueeze(1)
    w = w / w.mean()
  return (per_pixel * w).mean()


@torch.no_grad()
def pair_zorluk(m, bs=16):
    """Pair başına dist yaklaşığı (seg hata % + pose terimi). Yalnız örnekleme ağırlığı olduğundan
    resmi formülün pair'e bölünüşünden sapması önemsiz."""
    z = torch.zeros(NUM_PAIRS, device=device)
    for s0 in range(0, len(PILOT), bs):
        ids = PILOT[s0:s0 + bs]
        full = inflate(m(ids))
        pred = distortion_net.segnet(seg_preprocess_grad(full)).argmax(1)
        pose = distortion_net.posenet(pose_preprocess_grad(full))["pose"][:, :6].float()
        seg = 100 * (pred != seg_targets[ids].long()).float().mean((1, 2))
        z[ids] = seg + (10 * (pose - pose_targets[ids]).square().mean(1)).sqrt()
    return z[PILOT]


def l7_tarama(init_yol, ayarlar, tag, epochs=150, **kw):
    """Kısa net_ft_q turlarıyla l7 ayar taraması. Notlar donuk → KB sabit, karar saf `bits` bit dist'le.
    ayarlar: [("ad", dict(thr=2.0)), ("sinifw", dict(sinif_w=[1, 4, 1, 3, 1])), ...]"""
    global L7, SINIF_W
    eski, eski_w = dict(L7), SINIF_W
    try:
        for ad, ayar in ayarlar:
            ayar = dict(ayar)
            SINIF_W = (torch.tensor(ayar.pop("sinif_w"), dtype=torch.float32, device=device)
                       if "sinif_w" in ayar else eski_w)
            L7 = eski | ayar
            ek = f" | sınıf ağırlığı {SINIF_W.tolist()}" if SINIF_W is not None else ""
            print(f"\n--- l7 {ad}: {L7}{ek} ---")
            net_ft_q(init_yol, f"{tag}_{ad}", epochs=epochs, **kw)
    finally:
        L7, SINIF_W = eski, eski_w

## 15. (v6) Deneyler
Sıra önerisi: **v6-a** (dakikalar, kayıpsız kazanç) → **v6-b** raporu → l7/zor taramaları → gece kuyruğu.
devam2_ft600 bitince `YOL`'u ona çevir; notlar donuk kaldığından v6-a'nın ent'i ft taramalarında da geçerli.

In [ ]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [20, 20, 20, 20, 16, 16, 10]
mimari_kur(GRID_LEVELS, WIDTHS, 600)

# v6-a: kayıpsız ölçüm kazançları — zamansal entropi + delta kodlu kodlar. dist'e dokunmaz, dakikalar sürer.
YOL = OUT_DIR / "pilot_rate_p600u_film8_devam1_ft600.pt"       # devam2_ft600 bitince onu yaz
ent = ent_egit(YOL, "v6a", h=16, epochs=200)
olc_film2(YOL, k=8, bits=8, ent=ent)

[v6a] E050 | 1/64 22.8>18.1 | 1/32 6.1>6.3 | 1/16 16.5>14.1 KB | ent 6.0 KB | kazanç +0.9 KB (-0.0006 skor)
[v6a] E100 | 1/64 22.8>18.0 | 1/32 6.1>5.6 | 1/16 16.5>13.9 KB | ent 6.0 KB | kazanç +1.9 KB (-0.0013 skor)
[v6a] E150 | 1/64 22.8>18.0 | 1/32 6.1>5.4 | 1/16 16.5>13.9 KB | ent 6.0 KB | kazanç +2.1 KB (-0.0014 skor)
[v6a] E200 | 1/64 22.8>18.0 | 1/32 6.1>5.4 | 1/16 16.5>13.9 KB | ent 6.0 KB | kazanç +2.1 KB (-0.0014 skor)
dist (8 bit ressam + kod/film) 0.2150
notlar 43.2 (iid 45.4 / zamansal 43.2) + ressam 67.7 + kodlar 4.3 (iid 4.3 / delta 4.5) + film 2.3 = 117.6 KB | TOPLAM 0.293


In [ ]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [20, 20, 20, 20, 16, 16, 10]
mimari_kur(GRID_LEVELS, WIDTHS, 600)

# v6-b: seviye/kanal maliyet-katkı raporu (~9 skor ölçümü). Kârsız kanal çıkarsa alttaki kalıp.
FILM8 = lambda n, l, w: ModGridDecoder(n, l, w, k=8)
seviye_raporu(OUT_DIR / "pilot_rate_p600u_film8_devam1_ft600.pt", sinif=FILM8)

# kanal_sil(OUT_DIR / "pilot_rate_p600u_film8_devam1_ft600.pt", {64: [0], 32: [0], 16: [0, 1]}, "v6_kanal", sinif=FILM8)
# GRID_LEVELS = {64: 1, 32: 1, 16: 2}; mimari_kur(GRID_LEVELS, WIDTHS, 600)
# sonra cell 34'teki devam kalıbı: run_rate(init=pilot_rate_v6_kanal.pt, sinif=FILM8) + net_ft_q + olc_film2

taban dist 0.2149 | karar: Δdist < KB karşılığı → kârsız
1/16:  16.5 KB (skor 0.0110) | sıfır %98 | sıfırlanırsa Δdist +0.3707 
   kanal 0:   8.3 KB (skor 0.0055) | Δdist +0.1613 
   kanal 1:   8.2 KB (skor 0.0055) | Δdist +0.2119 
1/32:   6.1 KB (skor 0.0041) | sıfır %97 | sıfırlanırsa Δdist +0.1704 
   kanal 0:   3.3 KB (skor 0.0022) | Δdist +0.0915 
   kanal 1:   2.7 KB (skor 0.0018) | Δdist +0.0728 
1/64:  22.8 KB (skor 0.0152) | sıfır %34 | sıfırlanırsa Δdist +5.4696 
   kanal 0:  10.9 KB (skor 0.0072) | Δdist +2.5974 
   kanal 1:  11.8 KB (skor 0.0079) | Δdist +3.4119 


In [ ]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [20, 20, 20, 20, 16, 16, 10]
mimari_kur(GRID_LEVELS, WIDTHS, 600)

# v6-c: l7 taraması — notlar donuk, KB sabit → karar saf 8 bit dist. Ayar başına ~150 epoch.
FILM8 = lambda n, l, w: ModGridDecoder(n, l, w, k=8)
AYARLAR = [
    ("thr2",   dict(thr=2.0)),
    ("mult8",  dict(mult=8.0)),
    ("tau15",  dict(tau=0.15)),
    ("sinifw", dict(sinif_w=[1.0, 4.0, 1.0, 3.0, 1.0])),
]
l7_tarama(OUT_DIR / "pilot_rate_p600u_film8_devam1_ft600.pt", AYARLAR, "v6l7",
          epochs=150, bits=8, sinif=FILM8, kod_lr=1e-2)
# Kazanan ayarı L7 / SINIF_W'ye yazıp tam 600 epoch net_ft_q ile bitir (v6-d ile birleştirilebilir).


--- l7 thr2: {'tau': 0.3, 'thr': 2.0, 'mult': 4.0} ---
[v6l7_thr2] net_ft_q | QAT 8 bit | batch 40 | LR 6.71e-04 | notlar 45.4 KB | ressam başta 80.9 KB


  0%|          | 0/150 [00:00<?, ?it/s]

[v6l7_thr2] E100 | 8 bit dist 0.2468 (seg 0.2010) | ressam 81.0 KB | toplam 0.331 | en iyi 0.331 (E100)
[v6l7_thr2] E150 | 8 bit dist 0.2175 (seg 0.1922) | ressam 81.0 KB | toplam 0.302 | en iyi 0.302 (E150)
[v6l7_thr2] SON | en iyi E150 | 8 bit dist 0.2175 (seg 0.1922, pose 0.0252) | notlar 45.4 + ressam 81.0 KB | TOPLAM 0.302

--- l7 mult8: {'tau': 0.3, 'thr': 1.0, 'mult': 8.0} ---
[v6l7_mult8] net_ft_q | QAT 8 bit | batch 40 | LR 6.71e-04 | notlar 45.4 KB | ressam başta 80.9 KB


  0%|          | 0/150 [00:00<?, ?it/s]

[v6l7_mult8] E100 | 8 bit dist 0.2330 (seg 0.1959) | ressam 81.0 KB | toplam 0.317 | en iyi 0.317 (E100)
[v6l7_mult8] E150 | 8 bit dist 0.2150 (seg 0.1891) | ressam 81.0 KB | toplam 0.299 | en iyi 0.299 (E150)
[v6l7_mult8] SON | en iyi E150 | 8 bit dist 0.2150 (seg 0.1891, pose 0.0258) | notlar 45.4 + ressam 81.0 KB | TOPLAM 0.299

--- l7 tau15: {'tau': 0.15, 'thr': 1.0, 'mult': 4.0} ---
[v6l7_tau15] net_ft_q | QAT 8 bit | batch 40 | LR 6.71e-04 | notlar 45.4 KB | ressam başta 80.9 KB


  0%|          | 0/150 [00:00<?, ?it/s]

[v6l7_tau15] E100 | 8 bit dist 0.2388 (seg 0.1922) | ressam 81.0 KB | toplam 0.323 | en iyi 0.323 (E100)
[v6l7_tau15] E150 | 8 bit dist 0.2086 (seg 0.1840) | ressam 81.0 KB | toplam 0.293 | en iyi 0.293 (E150)
[v6l7_tau15] SON | en iyi E150 | 8 bit dist 0.2086 (seg 0.1840, pose 0.0245) | notlar 45.4 + ressam 81.0 KB | TOPLAM 0.293

--- l7 sinifw: {'tau': 0.3, 'thr': 1.0, 'mult': 4.0} | sınıf ağırlığı [1.0, 4.0, 1.0, 3.0, 1.0] ---
[v6l7_sinifw] net_ft_q | QAT 8 bit | batch 40 | LR 6.71e-04 | notlar 45.4 KB | ressam başta 80.9 KB


  0%|          | 0/150 [00:00<?, ?it/s]

[v6l7_sinifw] E100 | 8 bit dist 0.2984 (seg 0.2559) | ressam 81.0 KB | toplam 0.382 | en iyi 0.382 (E100)
[v6l7_sinifw] E150 | 8 bit dist 0.2751 (seg 0.2492) | ressam 81.0 KB | toplam 0.359 | en iyi 0.359 (E150)
[v6l7_sinifw] SON | en iyi E150 | 8 bit dist 0.2751 (seg 0.2492, pose 0.0259) | notlar 45.4 + ressam 81.0 KB | TOPLAM 0.359


In [ ]:
PILOT = torch.arange(600, device=device)
GRID_LEVELS = {64: 2, 32: 2, 16: 2}
WIDTHS = [20, 20, 20, 20, 16, 16, 10]
mimari_kur(GRID_LEVELS, WIDTHS, 600)

# v6-d: zor-pair örneklemeli uzun ince ayar. zor_beta=1 → zorlukla orantılı örnekleme (kırpılı).
FILM8 = lambda n, l, w: ModGridDecoder(n, l, w, k=8)
net_ft_q(OUT_DIR / "pilot_rate_p600u_film8_devam1_ft600.pt", "v6_zor_ft600", bits=8, sinif=FILM8,
         kod_lr=1e-2, zor_beta=1.0, zor_yenile=50)
olc_film2(OUT_DIR / "pilot_rate_v6_zor_ft600.pt", ent=OUT_DIR / "ent_temp_v6a.pt")   # notlar donuk → v6-a ent'i geçerli

[v6_zor_ft600] net_ft_q | QAT 8 bit | batch 40 | LR 6.71e-04 | notlar 45.4 KB | ressam başta 80.9 KB


  0%|          | 0/600 [00:00<?, ?it/s]

[v6_zor_ft600] E100 | 8 bit dist 0.2639 (seg 0.2137) | ressam 81.0 KB | toplam 0.348 | en iyi 0.348 (E100)
[v6_zor_ft600] E200 | 8 bit dist 0.2548 (seg 0.2091) | ressam 81.0 KB | toplam 0.339 | en iyi 0.339 (E200)
[v6_zor_ft600] E300 | 8 bit dist 0.2645 (seg 0.2045) | ressam 81.0 KB | toplam 0.349 | en iyi 0.339 (E200)
[v6_zor_ft600] E400 | 8 bit dist 0.2327 (seg 0.1963) | ressam 81.0 KB | toplam 0.317 | en iyi 0.317 (E400)
[v6_zor_ft600] E500 | 8 bit dist 0.2285 (seg 0.1925) | ressam 81.0 KB | toplam 0.313 | en iyi 0.313 (E500)
[v6_zor_ft600] E600 | 8 bit dist 0.2131 (seg 0.1888) | ressam 81.0 KB | toplam 0.297 | en iyi 0.297 (E600)
[v6_zor_ft600] SON | en iyi E600 | 8 bit dist 0.2131 (seg 0.1888, pose 0.0243) | notlar 45.4 + ressam 81.0 KB | TOPLAM 0.297
dist (8 bit ressam + kod/film) 0.2131
notlar 43.2 (iid 45.4 / zamansal 43.2) + ressam 67.7 + kodlar 4.3 (iid 4.3 / delta 4.5) + film 2.3 = 117.6 KB | TOPLAM 0.291


In [28]:
# v6-e: λ taraması + ızgara tahsisi varyantları — 120 pair test tezgâhı, gece kuyruğu (saatler).
# Yeni seviye düzenleri sıfırdan FP32 ister (fp32_ep); λ taraması mevcut kaynaktan devam eder.
# Kazanan, üretim mimarisinde (böl. 11 kalıbı) doğrulanmadan nihai sayılmaz.
sira = [
    dict(T="v6_lam20", lam=20, **SERT),
    dict(T="v6_lam45", lam=45, **SERT),
    dict(T="v6_c11_16x2", lam=30, levels={64: 1, 32: 1, 16: 2}, fp32="pilot_v6_c11_16x2.pt", fp32_ep=2400, **SERT),
T]
gece(sira)


=== başladı 2026-10-16 10:11 ===

[v6_lam20] PILOT 120 pair | ızgaralar {64: 2, 32: 2, 16: 2} | ressam [16, 16, 16, 16, 12, 12, 8] (45K parametre)
[v6_lam20] başlangıç pilot_p600_uzun.pt: FP32 dist 0.0733
[v6_lam20] adım 0.515 → pilot_rate_v6_lam20_init.pt
[v6_lam20] bit aşaması | parça 20 × birikim 1 = etkin batch 20 | 6 adım/epoch | not LR 1.65e-02 | ressam LR 4.74e-04


  0%|          | 0/1500 [00:00<?, ?it/s]

[v6_lam20] E250 λ=12 | dist 0.2847 | ızgara ~292 KB | sıfır %67/%54/%34 | toplam ≈ 0.509
[v6_lam20] E500 λ=20 | dist 0.3635 | ızgara ~146 KB | sıfır %89/%74/%47 | toplam ≈ 0.491
[v6_lam20] E750 λ=20 | dist 0.3832 | ızgara ~108 KB | sıfır %93/%80/%48 | toplam ≈ 0.486
[v6_lam20] E1000 λ=20 | dist 0.3598 | ızgara ~97 KB | sıfır %94/%83/%48 | toplam ≈ 0.454
[v6_lam20] E1250 λ=20 | dist 0.3441 | ızgara ~92 KB | sıfır %94/%85/%49 | toplam ≈ 0.436
[v6_lam20] E1500 λ=20 | dist 0.2613 | ızgara ~87 KB | sıfır %95/%85/%49 | toplam ≈ 0.350
[v6_lam20_ft600] net_ft | batch 40 | LR 6.71e-04 | EMA yok | QAT yok | notlar 87 KB | ressam 45 KB


  0%|          | 0/600 [00:00<?, ?it/s]

[v6_lam20_ft600] E050 | dist 0.2835 | 8 bit 0.2857 | ızgara 87 KB | toplam 0.374 | en iyi 0.2857 (E50)
[v6_lam20_ft600] E100 | dist 0.2509 | 8 bit 0.2577 | ızgara 87 KB | toplam 0.346 | en iyi 0.2577 (E100)
[v6_lam20_ft600] E150 | dist 0.2278 | 8 bit 0.2271 | ızgara 87 KB | toplam 0.315 | en iyi 0.2271 (E150)
[v6_lam20_ft600] E200 | dist 0.2164 | 8 bit 0.2200 | ızgara 87 KB | toplam 0.308 | en iyi 0.2200 (E200)
[v6_lam20_ft600] E250 | dist 0.2066 | 8 bit 0.2091 | ızgara 87 KB | toplam 0.297 | en iyi 0.2091 (E250)
[v6_lam20_ft600] E300 | dist 0.2032 | 8 bit 0.2055 | ızgara 87 KB | toplam 0.294 | en iyi 0.2055 (E300)
[v6_lam20_ft600] E350 | dist 0.1975 | 8 bit 0.1992 | ızgara 87 KB | toplam 0.288 | en iyi 0.1992 (E350)
[v6_lam20_ft600] E400 | dist 0.1932 | 8 bit 0.2037 | ızgara 87 KB | toplam 0.292 | en iyi 0.1992 (E350)
[v6_lam20_ft600] E450 | dist 0.1900 | 8 bit 0.1980 | ızgara 87 KB | toplam 0.286 | en iyi 0.1980 (E450)
[v6_lam20_ft600] E500 | dist 0.1882 | 8 bit 0.1926 | ızgara 87 KB

  0%|          | 0/1500 [00:00<?, ?it/s]

... (11 outputs left out)

  0%|          | 0/1500 [00:00<?, ?it/s]

[v6_c11_16x3] E250 λ=16 | dist 0.3549 | ızgara ~267 KB | sıfır %79/%61/%44 | toplam ≈ 0.563
[v6_c11_16x3] E500 λ=30 | dist 0.4741 | ızgara ~111 KB | sıfır %93/%74/%51 | toplam ≈ 0.578
[v6_c11_16x3] E750 λ=30 | dist 0.5122 | ızgara ~84 KB | sıfır %96/%79/%51 | toplam ≈ 0.598
[v6_c11_16x3] E1000 λ=30 | dist 0.4214 | ızgara ~76 KB | sıfır %96/%81/%49 | toplam ≈ 0.502
[v6_c11_16x3] E1250 λ=30 | dist 0.4006 | ızgara ~71 KB | sıfır %96/%82/%49 | toplam ≈ 0.478
[v6_c11_16x3] E1500 λ=30 | dist 0.3221 | ızgara ~67 KB | sıfır %97/%83/%50 | toplam ≈ 0.397
[v6_c11_16x3_ft600] net_ft | batch 40 | LR 6.71e-04 | EMA yok | QAT yok | notlar 67 KB | ressam 45 KB


  0%|          | 0/600 [00:00<?, ?it/s]

[v6_c11_16x3_ft600] E050 | dist 0.2945 | 8 bit 0.2953 | ızgara 67 KB | toplam 0.370 | en iyi 0.2953 (E50)
[v6_c11_16x3_ft600] E100 | dist 0.2983 | 8 bit 0.3043 | ızgara 67 KB | toplam 0.379 | en iyi 0.2953 (E50)
[v6_c11_16x3_ft600] E150 | dist 0.2753 | 8 bit 0.2810 | ızgara 67 KB | toplam 0.356 | en iyi 0.2810 (E150)
[v6_c11_16x3_ft600] E200 | dist 0.2616 | 8 bit 0.2604 | ızgara 67 KB | toplam 0.335 | en iyi 0.2604 (E200)
[v6_c11_16x3_ft600] E250 | dist 0.2590 | 8 bit 0.2602 | ızgara 67 KB | toplam 0.335 | en iyi 0.2602 (E250)
[v6_c11_16x3_ft600] E300 | dist 0.2483 | 8 bit 0.2518 | ızgara 67 KB | toplam 0.326 | en iyi 0.2518 (E300)
[v6_c11_16x3_ft600] E350 | dist 0.2503 | 8 bit 0.2566 | ızgara 67 KB | toplam 0.331 | en iyi 0.2518 (E300)
[v6_c11_16x3_ft600] E400 | dist 0.2460 | 8 bit 0.2468 | ızgara 67 KB | toplam 0.321 | en iyi 0.2468 (E400)
[v6_c11_16x3_ft600] E450 | dist 0.2395 | 8 bit 0.2417 | ızgara 67 KB | toplam 0.316 | en iyi 0.2417 (E450)
[v6_c11_16x3_ft600] E500 | dist 0.2366 